# 01b GDELT Event Explainer (Structured Evidence Module)

This notebook builds a research-grade, production-quality **event explanation layer** that sits next to the GDELT forecasting notebook.

It explains why a given day looks bullish, bearish, or important by extracting and summarizing structured event evidence from GDELT-like rows.

This notebook is **not** the final sentiment agent and **not** the final decision module. Its outputs are structured evidence packages designed for a later sentiment-agent fusion pipeline.

## 2) Scope and Non-Goals

### In Scope
- Event grouping from structured rows
- Event labeling (coarse + optional fine)
- Event-level sentiment, intensity, novelty, breadth, confidence scoring
- Daily explanation summaries and agent-ready exports
- Optional alignment with forecasting outputs

### Out of Scope
- Final multimodal fusion agent
- Autonomous agent actions or trading decisions
- End-to-end deep document modeling
- Financial advice

## 3) Design Summary: Complement to Forecasting Notebook

- The forecasting notebook is the **GDELT signal engine / forecaster**.
- This notebook is the **GDELT event explanation module**.
- A later sentiment agent will consume both:
  - Daily predictive signal outputs from the forecasting notebook
  - Structured event evidence exports from this notebook

## 4) Environment Checks

In [1]:
import importlib
import platform
import sys
import pandas as pd

def module_available(name: str) -> bool:
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False

packages = [
    "pandas",
    "numpy",
    "sklearn",
    "matplotlib",
    "sentence_transformers",
    "umap",
    "hdbscan",
    "rapidfuzz",
    "xgboost",
]

env_rows = [{"package": p, "available": module_available(p)} for p in packages]
env_df = pd.DataFrame(env_rows)

print(f"Python: {platform.python_version()}")
print(f"Executable: {sys.executable}")
print(f"pandas: {pd.__version__}")
print("\nPackage availability:")
print(env_df.to_string(index=False))

Python: 3.14.4
Executable: c:\Users\user\AppData\Local\Programs\Python\Python314\python.exe
pandas: 3.0.2

Package availability:
              package  available
               pandas       True
                numpy       True
              sklearn       True
           matplotlib       True
sentence_transformers      False
                 umap      False
              hdbscan      False
            rapidfuzz      False
              xgboost       True


## 5) Global Config

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path
import os
import pandas as pd


def discover_input_root() -> Path:
    env_input = os.environ.get("INPUT_DIR")
    if env_input:
        return Path(env_input)

    explicit_candidates = [
        Path.cwd() / "outputs" / "gdelt_structured_notebook",
        Path.home() / "Downloads" / "outputs" / "gdelt_structured_notebook",
    ]
    for cand in explicit_candidates:
        if (cand / "aggregates" / "section6_clean_df.parquet").exists():
            return cand

    for root in [Path.cwd(), Path.home()]:
        for cand in root.rglob("gdelt_structured_notebook"):
            if cand.is_dir() and (cand / "aggregates" / "section6_clean_df.parquet").exists():
                return cand

    return explicit_candidates[0]


@dataclass
class Config:
    PROJECT_ROOT: Path = Path(os.environ.get("PROJECT_ROOT", str(Path.cwd())))
    INPUT_DIR: Path = discover_input_root()
    OUTPUT_DIR: Path = Path(os.environ.get("OUTPUT_DIR", str(Path.cwd() / "outputs" / "gdelt_event_explainer")))

    CLEAN_ROW_PATH: Path = INPUT_DIR / "aggregates" / "section6_clean_df.parquet"
    DAILY_SIGNAL_PATH: Path = INPUT_DIR / "tables" / "leaderboard_forecasting_f3.csv"
    DAILY_PREDICTIONS_PATH: Path = INPUT_DIR / "predictions" / "forecast_holdout_predictions.csv"
    DAILY_FEATURES_PATH: Path = INPUT_DIR / "aggregates" / "gdelt_daily_aggregates.parquet"

    RUN_OPTIONAL_EMBEDDING_CLUSTERING: bool = False
    RUN_OPTIONAL_LLM_LABELING: bool = False
    TOP_K_THEMES: int = 30
    TOP_K_ORGS: int = 50
    TOP_K_LOCS: int = 50
    MIN_ROWS_PER_EVENT: int = 2
    MAX_EVENTS_PER_DAY_TO_EXPORT: int = 15
    EVENT_CLUSTER_MODE: str = "rule_based"
    EVENT_TIME_GRAIN: str = "day"
    EXPORT_JSON: bool = True
    EXPORT_PARQUET: bool = True
    EXPORT_CSV: bool = True
    RANDOM_SEED: int = 42
    COLAB_MODE: bool = False
    OPTIONAL_EMBEDDING_MODEL: str = "all-MiniLM-L6-v2"
    AGENT_READY_EXPORT: bool = True

    # Minimal export mode: skip writing most intermediate diagnostics to disk (keeps output folder clean).
    MINIMAL_EXPORTS: bool = os.environ.get("MINIMAL_EXPORTS", "true").strip().lower() in {"1","true","yes"}
    # Always write these filenames even in MINIMAL_EXPORTS mode.
    MINIMAL_EXPORT_ALLOWLIST: tuple[str, ...] = (
        "agent_ready_daily_event_payload.jsonl",
        "daily_event_explanations.json",
        "daily_event_packets_flat.csv",
        "event_detail_export.parquet",
        "event_detail_export.json",
        "event_quality_warnings.csv",
        "quality_check_table.csv",
        "final_validation_summary.csv",
        "final_publishability_validation.csv",
        "artifact_index.csv",
    )

    # Rule-based merge thresholds (deterministic defaults).
    THR_JACCARD_ORG: float = 0.20
    THR_JACCARD_LOC: float = 0.15
    THR_JACCARD_THEME: float = 0.20
    THR_JACCARD_URL: float = 0.10
    THR_DOMAIN_OVERLAP: float = 0.10
    THR_PAIR_SCORE: float = 0.42

    # Guardrails for debug-only surrogate data path.
    ALLOW_SURROGATE_MODE: bool = False

    # Mode/readiness thresholds.
    MIN_STRUCTURED_STRONG_RATE: float = 0.50
    MIN_STRUCTURED_SECONDARY_RATE: float = 0.20
    MIN_EVENT_CODE_NON_EMPTY_RATE: float = 0.005
    MIN_TEXT_NON_EMPTY_RATE: float = 0.01
    ALLOW_DEGRADED_FALLBACK: bool = True

    # Anti-collapse sanity thresholds for event granularity.
    MIN_MEDIAN_EVENTS_PER_NON_EMPTY_DAY: float = 2.0
    MIN_PCT_DAYS_GE2_EVENTS: float = 0.35
    MIN_PCT_DAYS_GE3_EVENTS: float = 0.15

CFG = Config()

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "artifacts").mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "artifacts" / "plots").mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "artifacts" / "reports").mkdir(parents=True, exist_ok=True)

print(pd.Series(asdict(CFG)).to_string())


PROJECT_ROOT                                                c:\Users\user\Desktop\client
INPUT_DIR                              c:\Users\user\Desktop\client\outputs\gdelt_str...
OUTPUT_DIR                             c:\Users\user\Desktop\client\outputs\gdelt_eve...
CLEAN_ROW_PATH                         c:\Users\user\Desktop\client\outputs\gdelt_str...
DAILY_SIGNAL_PATH                      c:\Users\user\Desktop\client\outputs\gdelt_str...
DAILY_PREDICTIONS_PATH                 c:\Users\user\Desktop\client\outputs\gdelt_str...
DAILY_FEATURES_PATH                    c:\Users\user\Desktop\client\outputs\gdelt_str...
RUN_OPTIONAL_EMBEDDING_CLUSTERING                                                  False
RUN_OPTIONAL_LLM_LABELING                                                          False
TOP_K_THEMES                                                                          30
TOP_K_ORGS                                                                            50
TOP_K_LOCS           

## 6) Reproducibility Helpers

In [3]:
import contextlib
import hashlib
import json
import random
import time
from typing import Any

import numpy as np
import pandas as pd

ARTIFACT_LOG: list[dict[str, Any]] = []

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def safe_mkdir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path

def log_artifact(path: Path, kind: str) -> None:
    ARTIFACT_LOG.append({"artifact": str(path), "type": kind})

def _should_write_artifact(path: Path) -> bool:
    if not getattr(CFG, "MINIMAL_EXPORTS", False):
        return True
    try:
        return path.name in set(getattr(CFG, "MINIMAL_EXPORT_ALLOWLIST", ()))
    except Exception:
        return True


def save_csv(df: pd.DataFrame, path: Path) -> None:
    if not _should_write_artifact(path):
        return
    safe_mkdir(path.parent)
    df.to_csv(path, index=False)
    log_artifact(path, "csv")

def save_parquet(df: pd.DataFrame, path: Path) -> None:
    if not _should_write_artifact(path):
        return
    safe_mkdir(path.parent)
    try:
        df.to_parquet(path, index=False)
        log_artifact(path, "parquet")
    except Exception as exc:
        print(f"[WARN] parquet save failed for {path.name}: {exc}")

def save_json(obj: Any, path: Path) -> None:
    if not _should_write_artifact(path):
        return
    safe_mkdir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=True, indent=2, default=str)
    log_artifact(path, "json")

def save_markdown(text: str, path: Path) -> None:
    if not _should_write_artifact(path):
        return
    safe_mkdir(path.parent)
    path.write_text(text, encoding="utf-8")
    log_artifact(path, "md")

def assert_columns(df: pd.DataFrame, required: list[str], name: str) -> None:
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"{name} missing columns: {missing}")

def null_audit(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    rows = []
    for c in cols:
        rows.append({"column": c, "null_share": float(df[c].isna().mean()) if c in df.columns else np.nan})
    return pd.DataFrame(rows)

def pretty_table(df: pd.DataFrame, n: int = 10) -> pd.DataFrame:
    return df.head(n)

@contextlib.contextmanager
def timed_block(label: str):
    t0 = time.perf_counter()
    yield
    dt = time.perf_counter() - t0
    print(f"[{label}] {dt:.2f}s")

def stable_hash(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8")).hexdigest()[:16]

seed_everything(CFG.RANDOM_SEED)
print("Helpers ready.")


Helpers ready.


## 7) Input Artifact Discovery

In [4]:
artifact_specs = [
    {"artifact_name": "clean_rows", "expected_path": CFG.CLEAN_ROW_PATH, "required_or_optional": "required"},
    {"artifact_name": "daily_signal_table", "expected_path": CFG.DAILY_SIGNAL_PATH, "required_or_optional": "optional"},
    {"artifact_name": "daily_predictions", "expected_path": CFG.DAILY_PREDICTIONS_PATH, "required_or_optional": "optional"},
    {"artifact_name": "daily_features", "expected_path": CFG.DAILY_FEATURES_PATH, "required_or_optional": "required"},
]

EVENT_RICH_ALIAS_GROUPS = {
    "themes": ["themes_clean", "themes"],
    "orgs": ["orgs_clean", "organizations"],
    "locs": ["locs_clean", "locations"],
    "event_code": ["event_code"],
    "event_root_code": ["event_root_code"],
    "event_base_code": ["event_base_code"],
    "actor1": ["actor1"],
    "actor2": ["actor2"],
    "action": ["action"],
    "goldstein_scale": ["goldstein_scale", "goldstein"],
    "quad_class": ["quad_class"],
}

def parquet_nested_to_pandas(path: Path) -> pd.DataFrame | None:
    """Robust parquet loader for nested/list-like columns via batch-wise column reads."""
    try:
        import pyarrow.parquet as pq
        pf = pq.ParquetFile(path)
        colnames = pf.schema.names
        out: dict[str, list[Any]] = {c: [] for c in colnames}

        for c in colnames:
            for batch in pf.iter_batches(columns=[c], batch_size=65536):
                arr = batch.column(0)
                try:
                    vals = arr.to_pylist()
                except Exception:
                    vals = [x.as_py() if hasattr(x, "as_py") else x for x in arr]
                out[c].extend(vals)

        return pd.DataFrame(out)
    except Exception as exc:
        print(f"[WARN] parquet nested fallback failed for {path.name}: {exc}")
        return None

def load_any_table(path: Path) -> tuple[pd.DataFrame | None, str]:
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path, low_memory=False), "csv"
    if suffix in {".pkl", ".pickle"}:
        return pd.read_pickle(path), "pickle"
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True), "jsonl"
    if suffix == ".json":
        try:
            obj = pd.read_json(path)
            return obj, "json"
        except Exception:
            with path.open("r", encoding="utf-8") as f:
                loaded = json.load(f)
            if isinstance(loaded, list):
                return pd.DataFrame(loaded), "json-list"
            return pd.DataFrame([loaded]), "json-object"
    if suffix == ".parquet":
        try:
            return pd.read_parquet(path), "parquet:pandas-default"
        except Exception as exc:
            print(f"[WARN] pandas parquet load failed for {path.name}: {exc}")
            if "Nested data conversions not implemented for chunked array outputs" in str(exc):
                print("[DIAG] Detected nested/chunked parquet conversion issue; trying nested-safe fallback loaders.")
            try:
                return pd.read_parquet(path, engine="pyarrow"), "parquet:pandas-pyarrow"
            except Exception as exc2:
                print(f"[WARN] explicit pyarrow parquet load failed for {path.name}: {exc2}")

            if path.stat().st_size > 400 * 1024 * 1024:
                print(f"[WARN] Skipping deep nested fallback for large parquet: {path.name}")
                return None, "failed-large-parquet"

            nested_df = parquet_nested_to_pandas(path)
            if isinstance(nested_df, pd.DataFrame):
                return nested_df, "parquet:pyarrow-batch-column-safe"
            return None, "failed"
    return None, "unsupported"

def find_clean_row_candidates() -> list[Path]:
    patterns = [
        "section6_clean_df.*",
        "aggregates/section6_clean_df.*",
        "aggregates/*clean*row*.parquet",
        "aggregates/*clean*row*.csv",
        "aggregates/*clean*row*.pkl",
        "aggregates/*clean*row*.pickle",
        "aggregates/*clean*row*.jsonl",
        "checkpoints/*row*.parquet",
        "tables/*row*export*.csv",
        "tables/*row*flat*.csv",
        "tables/*row_diagnostic*.csv",
    ]
    bases = [CFG.INPUT_DIR, CFG.PROJECT_ROOT / "outputs" / "gdelt_structured_notebook"]
    found: list[Path] = []
    for base in bases:
        if not base.exists():
            continue
        for pat in patterns:
            found.extend(base.glob(pat))
    dedup = []
    seen = set()
    for p in found:
        rp = str(p.resolve())
        if p.is_file() and rp not in seen:
            seen.add(rp)
            dedup.append(p)
    suffix_order = {".parquet": 0, ".csv": 1, ".pkl": 2, ".pickle": 3, ".jsonl": 4, ".json": 5}
    dedup.sort(key=lambda p: (suffix_order.get(p.suffix.lower(), 99), p.stat().st_size))
    return dedup

def candidate_event_richness(df: pd.DataFrame) -> tuple[int, float, dict[str, float]]:
    def is_effectively_empty_scalar(v: Any) -> bool:
        if v is None:
            return True
        if pd.isna(v):
            return True
        s = str(v).strip().lower()
        return s in {'', 'none', 'null', 'nan', 'na', '<na>', '[]', '{}', 'unknown', 'unknown_org', 'unknown_theme', 'unknown_loc'}

    def non_empty_token_rate(series: pd.Series) -> float:
        def one(v: Any) -> bool:
            if isinstance(v, (list, tuple, set)):
                return any(not is_effectively_empty_scalar(x) for x in v)
            return not is_effectively_empty_scalar(v)
        return float(series.map(one).mean()) if len(series) else 0.0

    rates: dict[str, float] = {}
    for canonical, aliases in EVENT_RICH_ALIAS_GROUPS.items():
        present = [a for a in aliases if a in df.columns]
        if not present:
            rates[canonical] = 0.0
            continue
        best = 0.0
        for c in present:
            s = df[c]
            non_empty = non_empty_token_rate(s)
            best = max(best, float(non_empty))
        rates[canonical] = best
    n_present = int(sum(v > 0 for v in rates.values()))
    avg_non_empty = float(np.mean(list(rates.values()))) if rates else 0.0
    return n_present, avg_non_empty, rates

inventory_rows = []
loaded_cache: dict[str, Any] = {}
loaded_cache_meta: dict[str, dict[str, Any]] = {}

for spec in artifact_specs:
    artifact_name = spec["artifact_name"]
    path = Path(spec["expected_path"])
    exists = path.exists()
    rows_loaded = np.nan
    obj = None
    loader_used = "none"
    resolved_path = str(path)

    if exists:
        obj, loader_used = load_any_table(path)
        if isinstance(obj, pd.DataFrame):
            rows_loaded = int(len(obj))
        else:
            print(f"[WARN] Could not load artifact into DataFrame: {path}")
    else:
        print(f"[WARN] Missing artifact: {path}")

    if artifact_name == "clean_rows" and not isinstance(obj, pd.DataFrame):
        best_pack = None
        for cand in find_clean_row_candidates():
            print(f"[DIAG] trying clean_rows candidate: {cand}")
            cand_df, cand_loader = load_any_table(cand)
            if not isinstance(cand_df, pd.DataFrame):
                continue

            n_present, avg_non_empty, rates = candidate_event_richness(cand_df)
            score = (n_present, avg_non_empty, int(len(cand_df)))
            cand_pack = {
                "df": cand_df,
                "loader": cand_loader,
                "path": cand,
                "n_present": n_present,
                "avg_non_empty": avg_non_empty,
                "rates": rates,
                "score": score,
            }
            if best_pack is None or cand_pack["score"] > best_pack["score"]:
                best_pack = cand_pack

        if best_pack is not None:
            obj = best_pack["df"]
            loader_used = best_pack["loader"]
            resolved_path = str(best_pack["path"])
            rows_loaded = int(len(obj))
            print(
                "[INFO] clean_rows resolved from best fallback: "
                f"{best_pack['path']} ({best_pack['loader']}) | "
                f"event_fields_present={best_pack['n_present']} avg_non_empty={best_pack['avg_non_empty']:.4f}"
            )
            loaded_cache_meta["clean_rows_candidate_quality"] = {
                "event_fields_present": int(best_pack["n_present"]),
                "avg_non_empty": float(best_pack["avg_non_empty"]),
                "rates": best_pack["rates"],
            }

    loaded_cache[artifact_name] = obj
    loaded_cache_meta[artifact_name] = {
        "source_path": resolved_path,
        "loader": loader_used,
        "is_surrogate": False,
    }
    inventory_rows.append({
        "artifact_name": artifact_name,
        "expected_path": str(path),
        "exists": bool(exists),
        "rows_loaded": rows_loaded,
        "required_or_optional": spec["required_or_optional"],
    })

input_inventory = pd.DataFrame(inventory_rows)
save_csv(input_inventory, CFG.OUTPUT_DIR / "artifacts" / "input_inventory.csv")

required_missing = input_inventory[(input_inventory["required_or_optional"] == "required") & ((~input_inventory["exists"]) | input_inventory["rows_loaded"].isna())]
if not required_missing.empty:
    missing_names = ", ".join(required_missing["artifact_name"].tolist())
    raise FileNotFoundError(
        f"Required official inputs are missing or unreadable: {missing_names}. "
        "Provide the clean-row and daily-feature artifacts via INPUT_DIR/PROJECT_ROOT env vars before running official exports.",
    )

threshold_table = pd.DataFrame([
    {"threshold_name": "THR_JACCARD_ORG", "value": CFG.THR_JACCARD_ORG},
    {"threshold_name": "THR_JACCARD_LOC", "value": CFG.THR_JACCARD_LOC},
    {"threshold_name": "THR_JACCARD_THEME", "value": CFG.THR_JACCARD_THEME},
    {"threshold_name": "THR_JACCARD_URL", "value": CFG.THR_JACCARD_URL},
    {"threshold_name": "THR_DOMAIN_OVERLAP", "value": CFG.THR_DOMAIN_OVERLAP},
    {"threshold_name": "THR_PAIR_SCORE", "value": CFG.THR_PAIR_SCORE},
])
save_csv(threshold_table, CFG.OUTPUT_DIR / "artifacts" / "cluster_thresholds_used.csv")

print("Input inventory:")
pretty_table(input_inventory, 20)

[WARN] pandas parquet load failed for section6_clean_df.parquet: Nested data conversions not implemented for chunked array outputs
[DIAG] Detected nested/chunked parquet conversion issue; trying nested-safe fallback loaders.
[WARN] explicit pyarrow parquet load failed for section6_clean_df.parquet: Nested data conversions not implemented for chunked array outputs
[WARN] Skipping deep nested fallback for large parquet: section6_clean_df.parquet
[WARN] Could not load artifact into DataFrame: c:\Users\user\Desktop\client\outputs\gdelt_structured_notebook\aggregates\section6_clean_df.parquet
[DIAG] trying clean_rows candidate: c:\Users\user\Desktop\client\outputs\gdelt_structured_notebook\section6_clean_df.parquet
[DIAG] trying clean_rows candidate: c:\Users\user\Desktop\client\outputs\gdelt_structured_notebook\aggregates\section6_clean_df.parquet
[WARN] pandas parquet load failed for section6_clean_df.parquet: Nested data conversions not implemented for chunked array outputs
[DIAG] Detect

,artifact_name,expected_path,exists,rows_loaded,required_or_optional
0,clean_rows,c:\Users\user\Desktop\client\outputs\gdelt_str...,True,683231,required
1,daily_signal_table,c:\Users\user\Desktop\client\outputs\gdelt_str...,True,8,optional
2,daily_predictions,c:\Users\user\Desktop\client\outputs\gdelt_str...,True,296,optional
3,daily_features,c:\Users\user\Desktop\client\outputs\gdelt_str...,True,1809,required


### Official Input Richness Gate

This hard gate prevents partial/low-information clean-row inputs from being treated as official production evidence.

In [5]:
clean_rows_loaded = loaded_cache.get("clean_rows")
if not isinstance(clean_rows_loaded, pd.DataFrame) or clean_rows_loaded.empty:
    raise RuntimeError("Official run blocked: clean_rows artifact is missing or empty after discovery.")

n_present_fields, avg_non_empty_rate, field_rates = candidate_event_richness(clean_rows_loaded)
richness_df = pd.DataFrame(
    [{"canonical_field": k, "non_empty_rate": float(v)} for k, v in field_rates.items()]
).sort_values("non_empty_rate", ascending=False)
save_csv(richness_df, CFG.OUTPUT_DIR / "artifacts" / "clean_rows_event_richness_gate.csv")

structured_non_empty = max(
    float(field_rates.get("themes", 0.0)),
    float(field_rates.get("orgs", 0.0)),
    float(field_rates.get("locs", 0.0)),
)

if n_present_fields < 3 or structured_non_empty < float(CFG.MIN_STRUCTURED_SECONDARY_RATE):
    raise RuntimeError(
        "Official run blocked: clean_rows does not meet minimum event-richness requirements "
        f"(fields_present={n_present_fields}, structured_non_empty={structured_non_empty:.4f}). "
        "Provide the canonical Notebook 1 clean-row artifact before exporting official outputs."
    )

print(
    f"Input richness gate passed: fields_present={n_present_fields}, "
    f"avg_non_empty_rate={avg_non_empty_rate:.4f}, structured_non_empty={structured_non_empty:.4f}"
)

RuntimeError: Official run blocked: clean_rows does not meet minimum event-richness requirements (fields_present=0, structured_non_empty=0.0000). Provide the canonical Notebook 1 clean-row artifact before exporting official outputs.

## 8) Raw Input Loading

In [5]:
df_rows = loaded_cache.get("clean_rows")
if not isinstance(df_rows, pd.DataFrame):
    raise RuntimeError("clean_rows artifact did not load as a DataFrame.")

raw_col_map = {}
row_input_resolution = {
    "source_path": str(loaded_cache_meta.get("clean_rows", {}).get("source_path", "<unknown>")),
    "loader": str(loaded_cache_meta.get("clean_rows", {}).get("loader", "<unknown>")),
    "n_input_rows": int(len(df_rows)),
    "n_input_columns": int(len(df_rows.columns)),
}


def normalize_list_like(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        arr = x
    elif isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = json.loads(s)
                arr = parsed if isinstance(parsed, list) else [s]
            except Exception:
                arr = [p.strip() for p in s.split(",")]
        else:
            arr = [p.strip() for p in s.split(",")]
    else:
        arr = [str(x)]
    out = []
    for v in arr:
        sv = str(v).strip()
        if sv and sv.lower() not in {"none", "null", "nan", "na", "<na>", ""}:
            out.append(sv)
    return out


def first_existing(df: pd.DataFrame, candidates: list[str], name: str, required: bool = False):
    for candidate in candidates:
        if candidate in df.columns:
            raw_col_map[name] = candidate
            return df[candidate]
    raw_col_map[name] = None
    if required:
        raise KeyError(f"Missing required column for {name}: {candidates}")
    return pd.Series([pd.NA] * len(df), index=df.index)


def non_empty_list_rate(series: pd.Series) -> float:
    return float(series.map(lambda x: len(normalize_list_like(x)) > 0).mean()) if len(series) else 0.0


themes_rate = non_empty_list_rate(df_rows["themes_clean"]) if "themes_clean" in df_rows.columns else 0.0
orgs_rate = non_empty_list_rate(df_rows["orgs_clean"]) if "orgs_clean" in df_rows.columns else 0.0
locs_rate = non_empty_list_rate(df_rows["locs_clean"]) if "locs_clean" in df_rows.columns else 0.0

# Notebook 2 is GKG-structured mode only; keep the other modes as disabled diagnostics.
event_code_mode = False
semantic_text_mode = False
SURROGATE_MODE_ACTIVE = False
MODE_FLAGS = {
    "event_code_mode": event_code_mode,
    "semantic_text_mode": semantic_text_mode,
    "surrogate_mode_active": SURROGATE_MODE_ACTIVE,
}
operating_mode = "GKG_structured_mode" if max(themes_rate, orgs_rate, locs_rate) >= CFG.MIN_STRUCTURED_STRONG_RATE else "degraded_debug_mode"

rows = pd.DataFrame(index=df_rows.index)
rows["date"] = first_existing(df_rows, ["date", "event_date"], "date", required=True)
rows["timestamp_primary"] = first_existing(df_rows, ["timestamp_primary", "timestamp_published", "timestamp_collected"], "timestamp_primary")
rows["source_domain"] = first_existing(df_rows, ["source_domain", "domain", "domain_clean"], "source_domain")
rows["url"] = first_existing(df_rows, ["url"], "url")
rows["clipped_tone"] = first_existing(df_rows, ["clipped_tone", "tone", "raw_tone"], "clipped_tone")
rows["tone_sign"] = first_existing(df_rows, ["tone_sign"], "tone_sign")
rows["themes_clean"] = first_existing(df_rows, ["themes_clean", "themes"], "themes_clean")
rows["orgs_clean"] = first_existing(df_rows, ["orgs_clean", "organizations"], "orgs_clean")
rows["locs_clean"] = first_existing(df_rows, ["locs_clean", "locations"], "locs_clean")
rows["event_code"] = first_existing(df_rows, ["event_code"], "event_code")
rows["event_root_code"] = first_existing(df_rows, ["event_root_code"], "event_root_code")
rows["event_base_code"] = first_existing(df_rows, ["event_base_code"], "event_base_code")
rows["actor1"] = first_existing(df_rows, ["actor1"], "actor1")
rows["actor2"] = first_existing(df_rows, ["actor2"], "actor2")
rows["action"] = first_existing(df_rows, ["action"], "action")
rows["goldstein_scale"] = first_existing(df_rows, ["goldstein_scale", "goldstein"], "goldstein_scale")
rows["quad_class"] = first_existing(df_rows, ["quad_class"], "quad_class")
rows["credibility_tier"] = first_existing(df_rows, ["credibility_tier", "source_credibility_tier"], "credibility_tier")
rows["title"] = first_existing(df_rows, ["title", "headline"], "title")
rows["headline"] = first_existing(df_rows, ["headline", "title"], "headline")
rows["snippet"] = first_existing(df_rows, ["snippet", "summary", "description"], "snippet")
rows["text"] = first_existing(df_rows, ["text", "content", "body"], "text")

rows["date"] = pd.to_datetime(rows["date"], errors="coerce", utc=True).dt.floor("D")
rows["timestamp_primary"] = pd.to_datetime(rows["timestamp_primary"], errors="coerce", utc=True)
rows["clipped_tone"] = pd.to_numeric(rows["clipped_tone"], errors="coerce")
rows["tone_sign"] = pd.to_numeric(rows["tone_sign"], errors="coerce")

# FIX: If URLs are missing/empty, construct synthetic but traceable URLs for representative evidence selection.
url_missing_rate = float(rows["url"].isna().mean()) if len(rows) else 0.0
if url_missing_rate > 0.90:
    print(f"[INFO] URL field missing in {url_missing_rate:.1%} of rows; constructing synthetic URLs for evidence tracing...")
    rows["url"] = rows.apply(
        lambda r: (
            r["url"]
            if pd.notna(r["url"]) and str(r["url"]).strip() not in {"", "<NA>", "nan", "none"}
            else f"local://{str(r['date'].date())}/{str(r['source_domain']) or 'unknown'}/{np.random.randint(0, 99999):05d}"
        ),
        axis=1,
    )

mode_report = pd.DataFrame([
    {"metric": "selected_mode", "value": operating_mode},
    {"metric": "themes_non_empty_rate", "value": themes_rate},
    {"metric": "orgs_non_empty_rate", "value": orgs_rate},
    {"metric": "locs_non_empty_rate", "value": locs_rate},
    {"metric": "event_code_mode_enabled", "value": int(event_code_mode)},
    {"metric": "semantic_text_mode_enabled", "value": int(semantic_text_mode)},
    {"metric": "surrogate_mode_active", "value": int(SURROGATE_MODE_ACTIVE)},
    {"metric": "url_missing_rate", "value": url_missing_rate},
])

save_csv(mode_report, CFG.OUTPUT_DIR / "artifacts" / "operating_mode_report.csv")
save_json(raw_col_map, CFG.OUTPUT_DIR / "artifacts" / "column_mapping.json")
save_json(row_input_resolution, CFG.OUTPUT_DIR / "artifacts" / "row_input_resolution.json")
save_json(MODE_FLAGS, CFG.OUTPUT_DIR / "artifacts" / "mode_flags.json")

print("Loaded rows shape:", rows.shape)
print("Row input resolution:", row_input_resolution)
print("Selected operating mode:", operating_mode)
print("Mode flags:", MODE_FLAGS)
print(f"URL recovery: {(1.0 - rows['url'].isna().mean()):.1%} of rows have usable URLs")
print("Column mapping:")
print(pd.Series(raw_col_map).to_string())

Loaded rows shape: (683231, 22)
Row input resolution: {'source_path': 'C:\\Users\\user\\Downloads\\outputs\\gdelt_structured_notebook\\aggregates\\section6_clean_df.parquet', 'loader': 'parquet:pandas-default', 'n_input_rows': 683231, 'n_input_columns': 39}
Selected operating mode: GKG_structured_mode
Mode flags: {'event_code_mode': False, 'semantic_text_mode': False, 'surrogate_mode_active': False}
URL recovery: 100.0% of rows have usable URLs
Column mapping:
date                              date
timestamp_primary    timestamp_primary
source_domain            source_domain
url                                url
clipped_tone              clipped_tone
tone_sign                    tone_sign
themes_clean              themes_clean
orgs_clean                  orgs_clean
locs_clean                  locs_clean
event_code                         NaN
event_root_code                    NaN
event_base_code                    NaN
actor1                             NaN
actor2                      

In [6]:
print("themes_clean dtype:", rows["themes_clean"].dtype)
print("themes_clean notna mean:", float(rows["themes_clean"].notna().mean()))
print("themes_clean sample:", rows["themes_clean"].head(3).tolist())
print("orgs_clean notna mean:", float(rows["orgs_clean"].notna().mean()))
print("locs_clean notna mean:", float(rows["locs_clean"].notna().mean()))

themes_clean dtype: str
themes_clean notna mean: 1.0
themes_clean sample: ['["IDEOLOGY", "ECON_WORLDCURRENCIES_ROUBLE", "GENERAL_GOVERNMENT", "EPU_POLICY_GOVERNMENT", "ECON_WORLDCURRENCIES_DOLLAR", "ECON_CURRENCY_EXCHANGE_RATE", "GENERAL_HEALTH", "HEALTH_VACCINATION", "WB_642_CHILD_HEALTH", "WB_1459_IMMUNIZATIONS", "WB_621_HEALTH_NUTRITION_AND_POPULATION", "WB_639_REPRODUCTIVE_MATERNAL_AND_CHILD_HEALTH", "UNGP_HEALTHCARE", "TAX_FNCACT_ECONOMISTS", "ECON_CENTRALBANK", "WB_1235_CENTRAL_BANKS", "WB_318_FINANCIAL_ARCHITECTURE_AND_BANKING", "WB_1920_FINANCIAL_SECTOR_DEVELOPMENT", "WB_1234_BANKING_INSTITUTIONS", "EPU_POLICY_CENTRAL_BANK", "TAX_FNCACT_PERFORMER", "VETO", "USPEC_UNCERTAINTY1", "ENV_OIL", "TAX_WORLDLANGUAGES_RUSSIA", "EPU_ECONOMY", "EPU_ECONOMY_HISTORIC", "HEALTH_PANDEMIC", "WB_635_PUBLIC_HEALTH", "WB_2165_HEALTH_EMERGENCIES", "WB_2166_HEALTH_EMERGENCY_PREPAREDNESS_AND_DISASTER_RESPONSE", "WB_2167_PANDEMICS", "TAX_DISEASE_CORONAVIRUS", "TAX_FNCACT_PEERS", "MEDICAL", "TAX_ETHNIC

In [7]:
diag_rates = pd.DataFrame([
    {"col": "themes_clean", "notna_rate": float(rows["themes_clean"].notna().mean()), "sample0": str(rows["themes_clean"].iloc[0])[:120]},
    {"col": "orgs_clean", "notna_rate": float(rows["orgs_clean"].notna().mean()), "sample0": str(rows["orgs_clean"].iloc[0])[:120]},
    {"col": "locs_clean", "notna_rate": float(rows["locs_clean"].notna().mean()), "sample0": str(rows["locs_clean"].iloc[0])[:120]},
])
save_csv(diag_rates, CFG.OUTPUT_DIR / "artifacts" / "diag_structured_col_samples.csv")
print(diag_rates.to_string(index=False))

         col  notna_rate                                                                                                                  sample0
themes_clean         1.0 ["IDEOLOGY", "ECON_WORLDCURRENCIES_ROUBLE", "GENERAL_GOVERNMENT", "EPU_POLICY_GOVERNMENT", "ECON_WORLDCURRENCIES_DOLLAR"
  orgs_clean         1.0                                                        ["Thomson Reuters Trust Principles", "European Union", "Reuters"]
  locs_clean         1.0           ["Russia", "Russian", "Turkey", "Brazil", "Turkish", "South Africa", "Hungary", "Bengaluru, Karnataka, India"]


## 9) Event-Relevant Schema Audit


In [8]:
def normalize_list_like(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        arr = x
    elif isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = json.loads(s)
                arr = parsed if isinstance(parsed, list) else [s]
            except Exception:
                arr = [p.strip() for p in s.split(",")]
        else:
            arr = [p.strip() for p in s.split(",")]
    else:
        arr = [str(x)]
    out = []
    for v in arr:
        sv = str(v).strip()
        if sv and sv.lower() not in {"none", "null", "nan", "na", "<na>", ""}:
            out.append(sv)
    return out

def non_empty_scalar_rate(series: pd.Series) -> float:
    bad = {"", "none", "null", "nan", "na", "<na>"}
    return float(series.astype("string").map(lambda x: str(x).strip().lower() not in bad).mean()) if len(series) else 0.0

def non_empty_list_rate(series: pd.Series) -> float:
    return float(series.map(lambda x: len(normalize_list_like(x)) > 0).mean()) if len(series) else 0.0

audit_specs = [
    ("themes_clean", "list"),
    ("orgs_clean", "list"),
    ("locs_clean", "list"),
    ("event_code", "scalar"),
    ("event_root_code", "scalar"),
    ("event_base_code", "scalar"),
    ("goldstein_scale", "scalar"),
    ("quad_class", "scalar"),
    ("actor1", "scalar"),
    ("actor2", "scalar"),
    ("action", "scalar"),
    ("title", "scalar"),
    ("headline", "scalar"),
    ("snippet", "scalar"),
    ("text", "scalar"),
]

schema_rows = []
for col, kind in audit_specs:
    if col not in rows.columns:
        schema_rows.append(
            {
                "column": col,
                "kind": kind,
                "present": False,
                "non_null_rate": 0.0,
                "non_empty_rate": 0.0,
                "non_empty_list_rate": np.nan,
            }
        )
        continue

    s = rows[col]
    nn = float(s.notna().mean())
    if kind == "list":
        ner = non_empty_list_rate(s)
        lrate = ner
    else:
        ner = non_empty_scalar_rate(s)
        lrate = np.nan
    schema_rows.append(
        {
            "column": col,
            "kind": kind,
            "present": True,
            "non_null_rate": nn,
            "non_empty_rate": ner,
            "non_empty_list_rate": lrate,
        }
    )

schema_audit = pd.DataFrame(schema_rows)

rows["url_hash_audit"] = rows["url"].astype("string").map(lambda x: stable_hash(x) if x and x != "<NA>" else pd.NA)
duplicate_url_rate = float(rows["url_hash_audit"].duplicated(keep=False).mean()) if len(rows) else np.nan

if "title" in rows.columns and rows["title"].notna().any():
    duplicate_title_rate = float(rows["title"].astype("string").duplicated(keep=False).mean())
elif "headline" in rows.columns and rows["headline"].notna().any():
    duplicate_title_rate = float(rows["headline"].astype("string").duplicated(keep=False).mean())
else:
    duplicate_title_rate = np.nan

rows["domain_clean_audit"] = rows["source_domain"].astype("string").str.lower().str.strip()
domain_daily = (
    rows.dropna(subset=["date", "domain_clean_audit"])
    .groupby(["date", "domain_clean_audit"]).size().rename("row_count").reset_index()
)
if len(domain_daily):
    day_total = domain_daily.groupby("date")["row_count"].sum().rename("day_total")
    domain_daily = domain_daily.merge(day_total, on="date", how="left")
    domain_daily["share"] = domain_daily["row_count"] / domain_daily["day_total"]
    domain_top = domain_daily.sort_values(["date", "share"], ascending=[True, False]).groupby("date", as_index=False).head(1)
else:
    domain_top = pd.DataFrame(columns=["date", "domain_clean_audit", "row_count", "day_total", "share"])

source_breadth = (
    rows.dropna(subset=["date"]).groupby("date", as_index=False).agg(
        n_rows=("date", "size"),
        n_unique_domains=("source_domain", pd.Series.nunique),
        n_unique_urls=("url", pd.Series.nunique),
    )
)
if len(source_breadth):
    source_breadth_stats = pd.DataFrame(
        [
            {"metric": "median_unique_domains_per_day", "value": float(source_breadth["n_unique_domains"].median())},
            {"metric": "p10_unique_domains_per_day", "value": float(source_breadth["n_unique_domains"].quantile(0.10))},
            {"metric": "median_unique_urls_per_day", "value": float(source_breadth["n_unique_urls"].median())},
            {"metric": "p10_unique_urls_per_day", "value": float(source_breadth["n_unique_urls"].quantile(0.10))},
        ]
    )
else:
    source_breadth_stats = pd.DataFrame(columns=["metric", "value"])

audit_summary = pd.DataFrame(
    [
        {"metric": "selected_mode", "value": operating_mode},
        {"metric": "duplicate_url_rate", "value": duplicate_url_rate},
        {"metric": "duplicate_title_rate", "value": duplicate_title_rate},
    ]
)

save_csv(schema_audit, CFG.OUTPUT_DIR / "artifacts" / "event_schema_audit.csv")
save_csv(domain_top, CFG.OUTPUT_DIR / "artifacts" / "per_day_domain_concentration_top1.csv")
save_csv(source_breadth, CFG.OUTPUT_DIR / "artifacts" / "source_breadth_by_day.csv")
save_csv(source_breadth_stats, CFG.OUTPUT_DIR / "artifacts" / "source_breadth_stats.csv")
save_csv(audit_summary, CFG.OUTPUT_DIR / "artifacts" / "schema_audit_summary.csv")

print("Schema audit complete.")
print("Selected mode:", operating_mode)
print("Duplicate URL rate:", duplicate_url_rate)
print("Duplicate title/headline rate:", duplicate_title_rate)
print("\nCoverage table:")
print(schema_audit.to_string(index=False))
print("\nSource breadth stats:")
print(source_breadth_stats.to_string(index=False))
pretty_table(domain_top, 15)

Schema audit complete.
Selected mode: GKG_structured_mode
Duplicate URL rate: 0.0023344959464661292
Duplicate title/headline rate: nan

Coverage table:
         column   kind  present  non_null_rate  non_empty_rate  non_empty_list_rate
   themes_clean   list     True            1.0        1.000000             1.000000
     orgs_clean   list     True            1.0        0.884405             0.884405
     locs_clean   list     True            1.0        0.949052             0.949052
     event_code scalar     True            0.0        0.000000                  NaN
event_root_code scalar     True            0.0        0.000000                  NaN
event_base_code scalar     True            0.0        0.000000                  NaN
goldstein_scale scalar     True            0.0        0.000000                  NaN
     quad_class scalar     True            0.0        0.000000                  NaN
         actor1 scalar     True            0.0        0.000000                  NaN
        

,date,domain_clean_audit,row_count,day_total,share
64,2021-01-01 00:00:00+00:00,exchangerates.org.uk,459,687,0.668122
219,2021-01-02 00:00:00+00:00,exchangerates.org.uk,64,182,0.351648
315,2021-01-03 00:00:00+00:00,exchangerates.org.uk,38,185,0.205405
516,2021-01-04 00:00:00+00:00,exchangerates.org.uk,76,571,0.133100
999,2021-01-05 00:00:00+00:00,reuters.com,57,531,0.107345
1131,2021-01-06 00:00:00+00:00,cfi.net.cn,28,518,0.054054
1463,2021-01-07 00:00:00+00:00,exchangerates.org.uk,74,588,0.125850
1756,2021-01-08 00:00:00+00:00,exchangerates.org.uk,108,548,0.197080
1978,2021-01-09 00:00:00+00:00,exchangerates.org.uk,90,237,0.379747
2105,2021-01-10 00:00:00+00:00,exchangerates.org.uk,132,265,0.498113


## 10) Event-Explanation Assumptions, Required Inputs, and Leakage Rules

- Explanations use only rows available by end of day $t$.
- No future-day information is used for day-$t$ event summaries.
- Event explanations are structured evidence summaries, not proven causes.
- News syndication and repeated URLs can inflate apparent importance.
- GDELT is rich but noisy; source breadth and concentration should be monitored.
- Official exports require the clean-row and daily-feature inputs to be present and readable; missing inputs now hard-fail instead of silently producing partial outputs.
- Surrogate/debug mode is for local inspection only and cannot write official agent-ready exports.
- Degraded debug mode is blocked from official exports; production artifacts require `GKG_structured_mode`.

## 11) Row Normalization for Event Explanation


In [9]:
def norm_token(x: Any) -> str:
    s = str(x).strip().lower()
    return "" if s in {"", "none", "null", "nan", "na", "<na>", "n/a", "unknown", "unk"} else s

def norm_list(x: Any, max_len: int) -> list[str]:
    arr = normalize_list_like(x)
    cleaned = []
    seen = set()
    for v in arr:
        nv = norm_token(v)
        if nv and nv not in seen:
            seen.add(nv)
            cleaned.append(nv)
        if len(cleaned) >= max_len:
            break
    cleaned.sort()
    return cleaned

def top_original_tokens(x: Any, max_len: int) -> list[str]:
    arr = normalize_list_like(x)
    out = []
    seen = set()
    for v in arr:
        raw = str(v).strip()
        key = norm_token(raw)
        if key and key not in seen:
            seen.add(key)
            out.append(raw)
        if len(out) >= max_len:
            break
    return out

ev = rows.copy()
ev = ev.dropna(subset=["date"]).copy()
ev["event_date"] = pd.to_datetime(ev["date"], errors="coerce", utc=True).dt.floor("D")

ev["theme_set"] = ev["themes_clean"].map(lambda x: norm_list(x, CFG.TOP_K_THEMES))
ev["org_set"] = ev["orgs_clean"].map(lambda x: norm_list(x, CFG.TOP_K_ORGS))
ev["loc_set"] = ev["locs_clean"].map(lambda x: norm_list(x, CFG.TOP_K_LOCS))

ev["theme_top_original"] = ev["themes_clean"].map(lambda x: top_original_tokens(x, 5))
ev["org_top_original"] = ev["orgs_clean"].map(lambda x: top_original_tokens(x, 5))
ev["loc_top_original"] = ev["locs_clean"].map(lambda x: top_original_tokens(x, 5))

ev["primary_theme"] = ev["theme_set"].map(lambda x: x[0] if len(x) else "unknown_theme")
ev["primary_org"] = ev["org_set"].map(lambda x: x[0] if len(x) else "unknown_org")
ev["primary_loc"] = ev["loc_set"].map(lambda x: x[0] if len(x) else "unknown_loc")

ev["domain_clean"] = ev["source_domain"].astype("string").map(norm_token).replace({"": "unknown_domain"})
ev["url_hash"] = ev["url"].astype("string").map(lambda x: stable_hash(x) if norm_token(x) else pd.NA)

for c in ["event_root_code", "event_base_code", "quad_class", "goldstein_scale", "event_code", "actor1", "actor2", "action"]:
    if c in ev.columns:
        if c == "goldstein_scale":
            ev[c] = pd.to_numeric(ev[c], errors="coerce")
        else:
            ev[c] = ev[c].astype("string").map(norm_token)

ev["event_signature_seed"] = (
    ev["event_date"].astype("string") + "|" +
    ev["primary_org"] + "|" +
    ev["primary_theme"] + "|" +
    ev["primary_loc"]
)

ev["row_event_preview"] = (
    "org=" + ev["primary_org"] +
    " | theme=" + ev["primary_theme"] +
    " | loc=" + ev["primary_loc"] +
    " | tone=" + ev["clipped_tone"].round(3).astype("string")
)

print("Normalized rows ready:", ev.shape)
pretty_table(ev[["event_date", "primary_org", "primary_theme", "primary_loc", "row_event_preview"]], 8)


Normalized rows ready: (683231, 38)


,event_date,primary_org,primary_theme,primary_loc,row_event_preview
0,2021-01-01 00:00:00+00:00,european union,econ_centralbank,"bengaluru, karnataka, india",org=european union | theme=econ_centralbank | ...
1,2021-01-01 00:00:00+00:00,federal reserve,crisislex_crisislexrec,australian,org=federal reserve | theme=crisislex_crisisle...
2,2021-01-03 00:00:00+00:00,reuters,black_market,cuba,org=reuters | theme=black_market | loc=cuba | ...
3,2021-01-03 00:00:00+00:00,european union,econ_emergingecon,americans,org=european union | theme=econ_emergingecon |...
4,2021-01-04 00:00:00+00:00,reuters,democracy,aussie,org=reuters | theme=democracy | loc=aussie | t...
5,2021-01-04 00:00:00+00:00,reuters,democracy,aussie,org=reuters | theme=democracy | loc=aussie | t...
6,2021-01-04 00:00:00+00:00,reuters,democracy,aussie,org=reuters | theme=democracy | loc=aussie | t...
7,2021-01-04 00:00:00+00:00,china foreign exchange trade system,affect,china,org=china foreign exchange trade system | them...


## 12) Event Candidate Generation

Candidate generation creates deterministic, controllable same-day anchors before clustering. This reduces combinatorial explosion and keeps grouping auditable.


In [10]:
from itertools import combinations

ev = ev.reset_index(drop=True)
ev["row_id"] = np.arange(len(ev)).astype(int)

PRIMARY_PRODUCTION_MODE = operating_mode
EVENT_CODE_MODE_ACTIVE = bool(MODE_FLAGS.get("event_code_mode", False))
SEMANTIC_TEXT_MODE_ACTIVE = bool(MODE_FLAGS.get("semantic_text_mode", False))
DEGRADED_DEBUG_MODE_ACTIVE = PRIMARY_PRODUCTION_MODE == "degraded_debug_mode"

if PRIMARY_PRODUCTION_MODE == "GKG_structured_mode":
    print("[MODE] GKG_structured_mode: structured co-occurrence is primary path.")
elif PRIMARY_PRODUCTION_MODE == "event_code_mode":
    print("[MODE] event_code_mode: event-code anchors available and enabled.")
elif PRIMARY_PRODUCTION_MODE == "semantic_text_mode":
    print("[MODE] semantic_text_mode: text-like fields available; semantic enhancement possible.")
else:
    print("[MODE] degraded_debug_mode: low-confidence debug fallback only.")

def build_candidate_keys(r: pd.Series) -> list[tuple[str, str]]:
    d = str(r["event_date"].date())
    keys: list[tuple[str, str]] = []

    themes = r["theme_set"] if isinstance(r["theme_set"], list) else []
    orgs = r["org_set"] if isinstance(r["org_set"], list) else []
    locs = r["loc_set"] if isinstance(r["loc_set"], list) else []

    # Primary GKG structured candidates.
    for t in themes[:3]:
        for o in orgs[:3]:
            keys.append((f"{d}|theme_org|{t}|{o}", "theme_org"))
        for l in locs[:3]:
            keys.append((f"{d}|theme_loc|{t}|{l}", "theme_loc"))

    for o in orgs[:3]:
        for l in locs[:3]:
            keys.append((f"{d}|org_loc|{o}|{l}", "org_loc"))

    for t1, t2 in list(combinations(themes[:4], 2))[:4]:
        keys.append((f"{d}|theme_pair|{t1}|{t2}", "theme_pair"))

    for o in orgs[:3]:
        keys.append((f"{d}|org_center|{o}", "org_center"))
    for l in locs[:3]:
        keys.append((f"{d}|loc_center|{l}", "loc_center"))

    if themes and orgs and locs:
        keys.append((f"{d}|theme_org_loc|{themes[0]}|{orgs[0]}|{locs[0]}", "theme_org_loc"))

    # Optional event-code candidates when truly available.
    if EVENT_CODE_MODE_ACTIVE:
        er = str(r.get("event_root_code", "")).strip().lower()
        eb = str(r.get("event_base_code", "")).strip().lower()
        ec = str(r.get("event_code", "")).strip().lower()
        if er and er not in {"", "unknown", "<na>"}:
            keys.append((f"{d}|event_root|{er}", "event_root"))
        if eb and eb not in {"", "unknown", "<na>"}:
            keys.append((f"{d}|event_base|{eb}", "event_base"))
        if ec and ec not in {"", "unknown", "<na>"}:
            keys.append((f"{d}|event_code|{ec}", "event_code"))

    # Debug-only degraded fallback path.
    if DEGRADED_DEBUG_MODE_ACTIVE:
        dom = str(r.get("domain_clean", "unknown_domain"))
        if dom and dom != "unknown_domain":
            tsgn = float(pd.to_numeric(r.get("tone_sign", 0.0), errors="coerce") or 0.0)
            tone_bucket = "pos" if tsgn > 0 else ("neg" if tsgn < 0 else "neu")
            keys.append((f"{d}|domain_tone|{dom}|{tone_bucket}", "domain_tone"))

    if not keys:
        po = r["primary_org"]
        pt = r["primary_theme"]
        pl = r["primary_loc"]
        keys.append((f"{d}|fallback|{po}|{pt}|{pl}", "fallback"))

    return list(dict.fromkeys(keys))[:16]

ev["candidate_pairs"] = ev.apply(build_candidate_keys, axis=1)
trusted_types = {"theme_org", "theme_loc", "org_loc", "theme_pair", "org_center", "loc_center", "theme_org_loc", "event_root", "event_base", "event_code"}
ev["high_quality_candidate"] = ev["candidate_pairs"].map(lambda xs: any(t in trusted_types for _, t in xs))

row_event_candidates = ev[["row_id", "event_date", "candidate_pairs"]].explode("candidate_pairs").dropna(subset=["candidate_pairs"]).copy()
row_event_candidates[["candidate_key", "candidate_type"]] = pd.DataFrame(row_event_candidates["candidate_pairs"].tolist(), index=row_event_candidates.index)
row_event_candidates = row_event_candidates.drop(columns=["candidate_pairs"])

cand_sizes = row_event_candidates.groupby(["candidate_key", "candidate_type"], as_index=False).agg(
    event_date=("event_date", "min"),
    n_rows=("row_id", "size"),
    n_unique_rows=("row_id", "nunique"),
)

resolved_rows = row_event_candidates["row_id"].nunique()
coverage_rate = float(resolved_rows / max(1, len(ev)))
unresolved_rate = float(1.0 - coverage_rate)
high_quality_fail_rate = float((~ev["high_quality_candidate"]).mean())

candidate_stats = pd.DataFrame([
    {"metric": "total_rows", "value": len(ev)},
    {"metric": "resolved_rows", "value": resolved_rows},
    {"metric": "coverage_rate", "value": coverage_rate},
    {"metric": "unresolved_rate", "value": unresolved_rate},
    {"metric": "high_quality_candidate_fail_rate", "value": high_quality_fail_rate},
    {"metric": "n_candidates", "value": cand_sizes["candidate_key"].nunique()},
    {"metric": "selected_mode", "value": PRIMARY_PRODUCTION_MODE},
])

row_candidate_quality = ev[["row_id", "event_date", "high_quality_candidate", "row_event_preview"]].copy()
row_candidate_quality["candidate_fail_reason"] = np.where(
    row_candidate_quality["high_quality_candidate"],
    "ok",
    "fallback_only_or_sparse_entities",
)

print(candidate_stats.to_string(index=False))
save_parquet(row_event_candidates, CFG.OUTPUT_DIR / "artifacts" / "row_event_candidates.parquet")
save_csv(cand_sizes, CFG.OUTPUT_DIR / "artifacts" / "event_candidate_sizes.csv")
save_csv(row_candidate_quality, CFG.OUTPUT_DIR / "artifacts" / "row_candidate_quality.csv")
pretty_table(cand_sizes.sort_values("n_rows", ascending=False), 10)

[MODE] GKG_structured_mode: structured co-occurrence is primary path.
                          metric               value
                      total_rows              683231
                   resolved_rows              683231
                   coverage_rate                 1.0
                 unresolved_rate                 0.0
high_quality_candidate_fail_rate                 0.0
                    n_candidates             5041257
                   selected_mode GKG_structured_mode


,candidate_key,candidate_type,event_date,n_rows,n_unique_rows
498,2021-01-01|theme_loc|econ_currency_exchange_ra...,theme_loc,2021-01-01 00:00:00+00:00,461,461
689,2021-01-01|theme_loc|econ_worldcurrencies_doll...,theme_loc,2021-01-01 00:00:00+00:00,461,461
1184,2021-01-01|theme_org|econ_currency_exchange_ra...,theme_org,2021-01-01 00:00:00+00:00,458,458
82,2021-01-01|org_loc|exchange rates|british,org_loc,2021-01-01 00:00:00+00:00,458,458
1374,2021-01-01|theme_org|econ_worldcurrencies_doll...,theme_org,2021-01-01 00:00:00+00:00,458,458
76,2021-01-01|org_loc|euro to dollar exchange rat...,org_loc,2021-01-01 00:00:00+00:00,458,458
1354,2021-01-01|theme_org|econ_worldcurrencies_brit...,theme_org,2021-01-01 00:00:00+00:00,458,458
1355,2021-01-01|theme_org|econ_worldcurrencies_brit...,theme_org,2021-01-01 00:00:00+00:00,458,458
1175,2021-01-01|theme_org|econ_currency_exchange_ra...,theme_org,2021-01-01 00:00:00+00:00,458,458
661,2021-01-01|theme_loc|econ_worldcurrencies_brit...,theme_loc,2021-01-01 00:00:00+00:00,458,458


## 13) Event Clustering


In [11]:
# Memory-safe clustering: use candidate-key token overlap without exploding row-level joins.
from collections import Counter

cand_summary = row_event_candidates.groupby(["event_date", "candidate_key", "candidate_type"], as_index=False).agg(
    n_rows=("row_id", "nunique"),
)


def key_tokens(candidate_key: str) -> set[str]:
    parts = str(candidate_key).split("|")
    return set([p for p in parts[2:] if p])


def jaccard_tokens(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 0.0
    return float(len(a & b) / max(1, len(a | b)))


class UnionFind:
    def __init__(self, items):
        self.parent = {x: x for x in items}

    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            if ra < rb:
                self.parent[rb] = ra
            else:
                self.parent[ra] = rb


MAX_PAIRWISE_CANDIDATES_PER_DAY = 500
MAX_PAIRS_EVALUATED_PER_DAY = 25000

cluster_rows = []
pair_diag_summary_rows = []
merge_reason_counter = Counter()

for d, sub in cand_summary.groupby("event_date"):
    sub_pair = sub.sort_values("n_rows", ascending=False).head(MAX_PAIRWISE_CANDIDATES_PER_DAY).copy()
    sub_direct = sub[~sub["candidate_key"].isin(sub_pair["candidate_key"])].copy()

    records = sub_pair.to_dict(orient="records")
    for r in records:
        r["token_set"] = key_tokens(r["candidate_key"])

    keys = [r["candidate_key"] for r in records]
    uf = UnionFind(keys)

    pairs_eval = 0
    pairs_merged = 0
    score_sum = 0.0
    score_max = 0.0
    nrec = len(records)

    for i in range(nrec):
        r1 = records[i]
        for j in range(i + 1, nrec):
            if pairs_eval >= MAX_PAIRS_EVALUATED_PER_DAY:
                break
            r2 = records[j]

            jac = jaccard_tokens(r1["token_set"], r2["token_set"])
            same_type = float(r1["candidate_type"] == r2["candidate_type"])
            size_bonus = float(min(r1["n_rows"], r2["n_rows"]) / max(1, max(r1["n_rows"], r2["n_rows"])))

            pair_score = 0.65 * jac + 0.15 * same_type + 0.20 * size_bonus
            should_merge = bool(pair_score >= 0.34)
            reasons = []
            if jac >= CFG.THR_JACCARD_THEME:
                reasons.append("token_overlap")
            if same_type > 0:
                reasons.append("same_candidate_type")

            if should_merge:
                uf.union(r1["candidate_key"], r2["candidate_key"])
                pairs_merged += 1
                for reason in reasons:
                    merge_reason_counter[reason] += 1

            score_sum += pair_score
            score_max = max(score_max, pair_score)
            pairs_eval += 1

        if pairs_eval >= MAX_PAIRS_EVALUATED_PER_DAY:
            break

    pair_diag_summary_rows.append({
        "event_date": d,
        "candidate_keys_considered": len(keys),
        "pairs_evaluated": pairs_eval,
        "pairs_merged": pairs_merged,
        "mean_pair_score": score_sum / max(1, pairs_eval),
        "max_pair_score": score_max,
        "pairs_truncated": int(pairs_eval >= MAX_PAIRS_EVALUATED_PER_DAY),
    })

    root_meta: dict[str, dict[str, Any]] = {}
    for r in records:
        root = uf.find(r["candidate_key"])
        meta = root_meta.setdefault(root, {"token_set": set(), "n_rows": 0})
        meta["token_set"].update(r["token_set"])
        meta["n_rows"] += int(r["n_rows"])

    for k in keys:
        root = uf.find(k)
        cluster_rows.append({"event_date": d, "candidate_key": k, "cluster_root": root})

    for _, r in sub_direct.iterrows():
        direct_tokens = key_tokens(r["candidate_key"])
        best_root = None
        best_score = 0.0
        for root, meta in root_meta.items():
            jac = jaccard_tokens(direct_tokens, meta["token_set"])
            size_bonus = float(min(int(r["n_rows"]), int(meta["n_rows"])) / max(1, max(int(r["n_rows"]), int(meta["n_rows"]))))
            score = 0.70 * jac + 0.30 * size_bonus
            if score > best_score:
                best_score = score
                best_root = root
        assigned_root = best_root if best_root is not None and best_score >= 0.30 else r["candidate_key"]
        cluster_rows.append({"event_date": d, "candidate_key": r["candidate_key"], "cluster_root": assigned_root})

candidate_to_cluster = pd.DataFrame(cluster_rows)
candidate_to_cluster["event_day"] = candidate_to_cluster["event_date"].dt.strftime("%Y%m%d")
candidate_to_cluster["event_id"] = candidate_to_cluster.apply(
    lambda r: f"E{r['event_day']}_{stable_hash(str(r['cluster_root']))[:8]}",
    axis=1,
)

type_priority = {
    "theme_org_loc": 0,
    "theme_org": 1,
    "theme_loc": 2,
    "org_loc": 3,
    "theme_pair": 4,
    "org_center": 5,
    "loc_center": 6,
    "event_root": 7,
    "event_base": 8,
    "event_code": 9,
    "domain_tone": 10,
    "fallback": 99,
}

row_cluster = row_event_candidates.merge(
    cand_summary[["event_date", "candidate_key", "n_rows", "candidate_type"]],
    on=["event_date", "candidate_key", "candidate_type"],
    how="left",
).merge(
    candidate_to_cluster[["event_date", "candidate_key", "event_id"]],
    on=["event_date", "candidate_key"],
    how="left",
)
row_cluster["type_priority"] = row_cluster["candidate_type"].map(type_priority).fillna(999).astype(int)
row_cluster = row_cluster.sort_values(["row_id", "type_priority", "n_rows", "candidate_key"], ascending=[True, True, False, True])
row_event_assign = row_cluster.drop_duplicates(subset=["row_id"], keep="first")[["row_id", "event_id"]]

pair_diag = pd.DataFrame(pair_diag_summary_rows)
merge_reason_counts = pd.DataFrame(
    [{"merge_reason": k, "count": v} for k, v in merge_reason_counter.items()]
).sort_values("count", ascending=False) if len(merge_reason_counter) else pd.DataFrame(columns=["merge_reason", "count"])

if len(pair_diag):
    merged_pairs_total = int(pair_diag["pairs_merged"].sum())
else:
    merged_pairs_total = 0

event_clusters_rows = ev.merge(row_event_assign, on="row_id", how="left")
event_clusters_rows = event_clusters_rows[event_clusters_rows["event_id"].notna()].copy()

event_clusters_summary = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
    n_rows=("row_id", "nunique"),
    n_unique_urls=("url_hash", "nunique"),
    n_unique_domains=("domain_clean", "nunique"),
)

cluster_size_dist = event_clusters_summary["n_rows"].value_counts().sort_index().rename_axis("cluster_size").reset_index(name="count")
daily_event_dist = event_clusters_summary.groupby("event_date", as_index=False).size().rename(columns={"size": "n_events"})
singleton_rate = float((event_clusters_summary["n_rows"] == 1).mean()) if len(event_clusters_summary) else np.nan
unresolved_row_rate = float(1.0 - row_event_assign["event_id"].notna().mean()) if len(row_event_assign) else 0.0

top_merged_clusters = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
    n_rows=("row_id", "nunique"),
    orgs=("org_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:5]),
    themes=("theme_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:5]),
    locs=("loc_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:5]),
).sort_values(["n_rows", "event_date"], ascending=[False, False]).head(20)

save_parquet(event_clusters_rows, CFG.OUTPUT_DIR / "artifacts" / "event_clusters_rows.parquet")
save_csv(event_clusters_summary, CFG.OUTPUT_DIR / "artifacts" / "event_clusters_summary.csv")
save_csv(cluster_size_dist, CFG.OUTPUT_DIR / "artifacts" / "cluster_size_distribution.csv")
save_csv(daily_event_dist, CFG.OUTPUT_DIR / "artifacts" / "daily_event_count_distribution.csv")
save_csv(pair_diag, CFG.OUTPUT_DIR / "artifacts" / "cluster_pair_diagnostics.csv")
save_csv(merge_reason_counts, CFG.OUTPUT_DIR / "artifacts" / "merge_reason_counts.csv")
save_csv(top_merged_clusters, CFG.OUTPUT_DIR / "artifacts" / "top_merged_clusters_preview.csv")

print(f"Event clusters built: {event_clusters_summary['event_id'].nunique():,} | singleton rate={singleton_rate:.4f} | unresolved_row_rate={unresolved_row_rate:.4f}")
print(f"Merging thresholds: pair_score >= {0.34:.2f} | max candidates per day: {MAX_PAIRWISE_CANDIDATES_PER_DAY}")
print(f"Pairs merged total: {merged_pairs_total:,}")
pretty_table(top_merged_clusters, 20)

Event clusters built: 129,015 | singleton rate=0.6515 | unresolved_row_rate=0.0000
Merging thresholds: pair_score >= 0.34 | max candidates per day: 500
Pairs merged total: 2,683,057


,event_date,event_id,n_rows,orgs,themes,locs
26055,2022-02-28 00:00:00+00:00,E20220228_a7ff5334,1032,"[a bank, a bank african, a bank austrian raiff...","[act_harmthreaten, act_makestatement, act_yiel...","[afghan, afghanistan, al masraf al markazi, di..."
35197,2022-07-12 00:00:00+00:00,E20220712_4d227ad7,1031,"[a bank, a brookings institution, a central ba...","[act_harmthreaten, affect, agriculture, aid_hu...","[albacete, castilla-la manacha, spain, algeria..."
41260,2022-09-26 00:00:00+00:00,E20220926_93da644a,892,"[a bank, a bank switzerland, a bank to england...","[act_forceposture, act_makestatement, act_yiel...","[aleppo, ?alab, syria, america, american, amer..."
35509,2022-07-13 00:00:00+00:00,E20220713_bf9880b6,772,"[a analysts agency, a analysts in commerzbank ...","[act_makestatement, affect, agriculture, allia...","[afghanistan, algeria, america, american, amer..."
25737,2022-02-24 00:00:00+00:00,E20220224_c461c06c,754,"[a bank, a central bank, a central bank russia...","[act_forceposture, act_harmthreaten, affect, a...","[afghanistan, alfa, tatarstan, russia, almetye..."
111845,2025-04-11 00:00:00+00:00,E20250411_1d6e9234,706,"[a a as embassy china, a bank, a bank folk to ...","[affect, agriculture, appointment, armedconfli...","[albacete, castilla-la manacha, spain, aleppo,..."
111232,2025-04-03 00:00:00+00:00,E20250403_f4e54dd5,680,"[a branch, a central bank european, a departme...","[act_makestatement, affect, agriculture, allia...","[algeria, america, american, americans, amster..."
36016,2022-07-18 00:00:00+00:00,E20220718_935d2b53,676,"[a central bank, a chu, a european union, a fo...","[act_makestatement, affect, agriculture, aid_h...","[afghanistan, al masraf al markazi, dimashq, s..."
100956,2024-11-06 00:00:00+00:00,E20241106_1590725c,668,"[a bank, a campaign, a central bank, a committ...","[act_forceposture, affect, agriculture, allian...","[abu dhabi, abu zÂ¸aby, united arab emirates, a..."
38689,2022-08-23 00:00:00+00:00,E20220823_130739b6,659,"[a bank, a bank to england, a central bank, a ...","[act_makestatement, affect, agriculture, aid_h...","[-squirrel hill, florida, united states, afgha..."


## 13b) Optional Semantic Clustering (Colab Pro Enhancement)

This section is optional and OFF by default. It provides a semantic comparison layer against rule-based clusters for research diagnostics only.

Canonical outputs in this notebook remain rule-based deterministic clusters.


In [12]:
import importlib

semantic_cluster_comparison = pd.DataFrame()

text_signal_rate = max(
    float(non_empty_scalar_rate(rows["title"])) if "title" in rows.columns else 0.0,
    float(non_empty_scalar_rate(rows["headline"])) if "headline" in rows.columns else 0.0,
    float(non_empty_scalar_rate(rows["snippet"])) if "snippet" in rows.columns else 0.0,
    float(non_empty_scalar_rate(rows["text"])) if "text" in rows.columns else 0.0,
)

if MODE_FLAGS.get("semantic_text_mode", False) and CFG.RUN_OPTIONAL_EMBEDDING_CLUSTERING and text_signal_rate >= CFG.MIN_TEXT_NON_EMPTY_RATE:
    try:
        SentenceTransformer = importlib.import_module("sentence_transformers").SentenceTransformer
        sem = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
            orgs=("org_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:4]),
            themes=("theme_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:4]),
            locs=("loc_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:4]),
            snippet_text=("snippet", lambda s: " ".join([str(x) for x in s.astype("string").head(3) if str(x) not in {"<NA>", "", "nan"}])),
            title_text=("title", lambda s: " ".join([str(x) for x in s.astype("string").head(3) if str(x) not in {"<NA>", "", "nan"}])),
        )
        sem["semantic_text"] = sem.apply(
            lambda r: " | ".join([
                "title=" + r["title_text"],
                "snippet=" + r["snippet_text"],
                "orgs=" + ",".join(r["orgs"]),
                "themes=" + ",".join(r["themes"]),
                "locs=" + ",".join(r["locs"]),
            ]), axis=1
        )

        model = SentenceTransformer(CFG.OPTIONAL_EMBEDDING_MODEL)
        emb = model.encode(sem["semantic_text"].tolist(), show_progress_bar=False)
        sem["semantic_norm"] = np.linalg.norm(emb, axis=1)
        sem["semantic_cluster_hint"] = pd.qcut(sem["semantic_norm"], q=min(8, max(2, sem["semantic_norm"].nunique())), duplicates="drop")
        semantic_cluster_comparison = sem[["event_date", "event_id", "semantic_text", "semantic_cluster_hint"]].copy()
        save_csv(semantic_cluster_comparison, CFG.OUTPUT_DIR / "artifacts" / "semantic_cluster_comparison.csv")
        print("Optional semantic comparison built (text-backed).")
    except Exception as exc:
        print(f"[WARN] Optional semantic clustering failed: {exc}")
else:
    print("Semantic text clustering skipped: text-like fields unavailable or optional semantic mode disabled.")

Semantic text clustering skipped: text-like fields unavailable or optional semantic mode disabled.


## 14) Event Labeling


In [13]:
if globals().get("SURROGATE_MODE_ACTIVE", False):
    print("[WARN] SURROGATE_MODE_ACTIVE=True. Label outputs are debug-only until real row-level clean_rows load succeeds.")

label_rules = {
    "macro_economy_or_policy": ["inflation", "rates", "central_bank", "policy", "government", "gdp", "monetary"],
    "company_or_sector_news": ["earnings", "revenue", "guidance", "company", "sector", "manufacturing", "technology", "finance"],
    "geopolitics_or_conflict": ["conflict", "war", "sanction", "military", "border", "geopolitics", "diplomacy"],
    "legal_or_regulatory": ["law", "regulation", "court", "lawsuit", "probe", "investigation", "compliance", "fine"],
}

def score_rule_hits(token_text: str) -> dict[str, list[str]]:
    hits: dict[str, list[str]] = {}
    for label, kws in label_rules.items():
        matched = [kw for kw in kws if kw in token_text]
        if matched:
            hits[label] = matched
    return hits

def label_event(row: pd.Series) -> tuple[str, str, float, str, str]:
    token_fields = ["top_theme_tokens", "top_org_tokens", "top_loc_tokens", "event_code_tokens", "actor_action_tokens", "text_tokens"]
    field_tokens = {}
    for c in token_fields:
        vals = row.get(c, [])
        if isinstance(vals, list):
            field_tokens[c] = [str(v).lower() for v in vals if str(v).strip()]
        else:
            field_tokens[c] = []

    token_text = " ".join([t for arr in field_tokens.values() for t in arr])
    rule_hits = score_rule_hits(token_text)

    best_label = "market_commentary_or_misc"
    best_hits = []
    if rule_hits:
        best_label, best_hits = sorted(rule_hits.items(), key=lambda kv: (len(kv[1]), kv[0]), reverse=True)[0]

    # Secondary, weak fallback from structured entities when no direct rule match.
    if best_label == "market_commentary_or_misc":
        has_org = len(field_tokens["top_org_tokens"]) > 0
        has_loc = len(field_tokens["top_loc_tokens"]) > 0
        has_theme = len(field_tokens["top_theme_tokens"]) > 0
        if has_theme and has_org:
            best_label = "company_or_sector_news"
        elif has_loc and any(k in token_text for k in ["ministry", "government", "state"]):
            best_label = "macro_economy_or_policy"

    fine_label = f"rule_based::{best_label}"
    evidence_fields = [k for k, v in field_tokens.items() if len(v) > 0]
    ambiguity_penalty = 0.10 if len(rule_hits) > 2 else 0.0
    confidence = float(min(1.0, max(0.05, 0.28 + 0.12 * len(best_hits) + 0.05 * len(evidence_fields) - ambiguity_penalty)))

    return (
        best_label,
        fine_label,
        confidence,
        ",".join(evidence_fields),
        json.dumps(rule_hits if rule_hits else {}),
    )

event_labels_base = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
    top_theme_tokens=("theme_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:25]),
    top_org_tokens=("org_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:25]),
    top_loc_tokens=("loc_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:25]),
    event_code_tokens=("event_root_code", lambda s: sorted({x for x in s if isinstance(x, str) and x not in {"", "unknown", "<na>"}})[:10]),
    actor_action_tokens=("action", lambda s: sorted({x for x in s if isinstance(x, str) and x not in {"", "unknown", "<na>"}})[:8]),
    text_tokens=("snippet", lambda s: sorted({str(x).lower() for x in s.astype("string") if str(x) not in {"<NA>", "", "nan"}})[:8]),
)

labeled = event_labels_base.copy()
labeled[["coarse_event_label", "fine_event_label", "label_confidence_rule_based", "label_evidence_fields", "label_rule_hits"]] = labeled.apply(
    lambda r: pd.Series(label_event(r)), axis=1
)

coverage_value = float(labeled["coarse_event_label"].notna().mean()) if len(labeled) else 0.0
label_coverage = pd.DataFrame([
    {"metric": "n_events", "value": int(len(labeled))},
    {"metric": "label_coverage_rate", "value": coverage_value},
    {"metric": "selected_mode", "value": PRIMARY_PRODUCTION_MODE},
])
label_distribution = labeled["coarse_event_label"].value_counts(dropna=False).rename_axis("coarse_event_label").reset_index(name="count")
label_overlap_like = labeled.groupby(["coarse_event_label", "fine_event_label"], as_index=False).size().rename(columns={"size": "n_events"})

save_csv(label_coverage, CFG.OUTPUT_DIR / "artifacts" / "event_label_coverage.csv")
save_csv(label_distribution, CFG.OUTPUT_DIR / "artifacts" / "event_label_distribution.csv")
save_csv(label_overlap_like, CFG.OUTPUT_DIR / "artifacts" / "event_label_overlap_table.csv")

pretty_table(label_distribution, 20)

,coarse_event_label,count
0,macro_economy_or_policy,94160
1,company_or_sector_news,18274
2,geopolitics_or_conflict,7657
3,legal_or_regulatory,6597
4,market_commentary_or_misc,2327


## 15) Event-Level Scoring


In [14]:
def entropy_from_counts(counts: pd.Series) -> float:
    if counts.empty:
        return 0.0
    p = counts / max(1, counts.sum())
    return float(-(p * np.log(np.clip(p, 1e-12, None))).sum())

def normalize01(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").fillna(0.0)
    lo, hi = float(s.min()), float(s.max())
    if hi - lo < 1e-12:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)

def confidence_bucket(v: float) -> str:
    if v >= 0.75:
        return "high"
    if v >= 0.45:
        return "medium"
    return "low"

def concentration_index(items: list[str]) -> float:
    if not items:
        return 1.0
    vc = pd.Series(items).value_counts(normalize=True)
    return float((vc ** 2).sum())

event_metrics = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
    n_rows=("row_id", "nunique"),
    n_unique_urls=("url_hash", lambda s: int(s.astype("string").replace("<NA>", pd.NA).dropna().nunique())),
    n_unique_domains=("domain_clean", "nunique"),
    n_unique_orgs=("org_set", lambda s: len({x for arr in s if isinstance(arr, list) for x in arr})),
    n_unique_locs=("loc_set", lambda s: len({x for arr in s if isinstance(arr, list) for x in arr})),
    n_unique_themes=("theme_set", lambda s: len({x for arr in s if isinstance(arr, list) for x in arr})),
    avg_tone=("clipped_tone", "mean"),
    median_tone=("clipped_tone", "median"),
    tone_std=("clipped_tone", "std"),
    positive_share=("tone_sign", lambda s: float((s > 0).mean())),
    negative_share=("tone_sign", lambda s: float((s < 0).mean())),
    neutral_share=("tone_sign", lambda s: float((s == 0).mean())),
)
event_metrics["tone_std"] = event_metrics["tone_std"].fillna(0.0)
event_metrics["net_sentiment"] = event_metrics["positive_share"] - event_metrics["negative_share"]

structure_rows = []
for (d, eid), sub in event_clusters_rows.groupby(["event_date", "event_id"]):
    dom_vc = sub["domain_clean"].astype("string").value_counts()
    theme_items = [x for arr in sub["theme_set"] if isinstance(arr, list) for x in arr]
    org_items = [x for arr in sub["org_set"] if isinstance(arr, list) for x in arr]
    loc_items = [x for arr in sub["loc_set"] if isinstance(arr, list) for x in arr]
    url_non_null = sub["url_hash"].astype("string").replace("<NA>", pd.NA).dropna()
    duplicate_heavy = float(1.0 - (url_non_null.nunique() / max(1, len(url_non_null)))) if len(url_non_null) else 0.0

    theme_conc = concentration_index(theme_items)
    org_conc = concentration_index(org_items)
    entity_conc = np.nanmean([concentration_index(org_items), concentration_index(loc_items)]) if (org_items or loc_items) else 1.0

    structured_overlap_density = float((len(set(theme_items)) + len(set(org_items)) + len(set(loc_items))) / max(1, len(sub) * 3))
    structure_rows.append(
        {
            "event_date": d,
            "event_id": eid,
            "domain_entropy": entropy_from_counts(dom_vc),
            "domain_top_share": float(dom_vc.iloc[0] / max(1, dom_vc.sum())) if len(dom_vc) else 1.0,
            "theme_concentration": float(theme_conc),
            "entity_concentration": float(entity_conc),
            "duplicate_evidence_rate": float(duplicate_heavy),
            "structured_cohesion_raw": float(structured_overlap_density),
        }
    )

structure_df = pd.DataFrame(structure_rows)
event_metrics = event_metrics.merge(structure_df, on=["event_date", "event_id"], how="left")

event_tok = event_clusters_rows.groupby(["event_date", "event_id"], as_index=False).agg(
    token_signature=("event_signature_seed", lambda s: sorted(set(s.astype("string")))[:80]),
    primary_org=("primary_org", lambda s: s.dropna().iloc[0] if len(s.dropna()) else "unknown_org"),
    primary_theme=("primary_theme", lambda s: s.dropna().iloc[0] if len(s.dropna()) else "unknown_theme"),
)
event_tok = event_tok.sort_values(["event_date", "event_id"]).reset_index(drop=True)

history_by_key: dict[str, list[set[str]]] = {}
nov_scores = []
for _, r in event_tok.iterrows():
    key = f"{r['primary_org']}|{r['primary_theme']}"
    cur = set(r["token_signature"] if isinstance(r["token_signature"], list) else [])
    hist = history_by_key.get(key, [])
    sims = []
    for h in hist[-180:]:
        inter = len(cur & h)
        union = len(cur | h)
        sims.append(inter / union if union else 0.0)
    max_sim = max(sims) if sims else 0.0
    nov_scores.append(1.0 - max_sim)
    hist.append(cur)
    history_by_key[key] = hist
event_tok["novelty_score"] = nov_scores

event_metrics = event_metrics.merge(event_tok[["event_date", "event_id", "novelty_score"]], on=["event_date", "event_id"], how="left")

event_metrics["source_breadth"] = normalize01(np.log1p(event_metrics["n_unique_urls"]) + np.log1p(event_metrics["n_unique_domains"]))
event_metrics["theme_diversity"] = normalize01(event_metrics["n_unique_themes"])
event_metrics["entity_diversity"] = normalize01(event_metrics["n_unique_orgs"] + event_metrics["n_unique_locs"])
event_metrics["cohesion_score"] = normalize01(event_metrics["structured_cohesion_raw"].fillna(0.0))
event_metrics["novelty_score"] = normalize01(event_metrics["novelty_score"].fillna(0.0))
event_metrics["tone_extremeness"] = normalize01(event_metrics["avg_tone"].abs().fillna(0.0) + event_metrics["tone_std"].fillna(0.0))

event_metrics["impact_score_raw"] = (
    0.42 * normalize01(np.log1p(event_metrics["n_rows"])) +
    0.28 * event_metrics["source_breadth"] +
    0.20 * event_metrics["tone_extremeness"] +
    0.10 * event_metrics["novelty_score"]
)

penalty_small_cluster = np.where(event_metrics["n_rows"] < CFG.MIN_ROWS_PER_EVENT, 0.20, 0.0)
penalty_single_domain = np.where(event_metrics["domain_top_share"].fillna(1.0) > 0.80, 0.18, 0.0)
penalty_duplicate = np.clip(event_metrics["duplicate_evidence_rate"].fillna(0.0), 0.0, 0.20)

event_scores = event_metrics.merge(
    labeled[["event_date", "event_id", "coarse_event_label", "fine_event_label", "label_confidence_rule_based", "label_evidence_fields", "label_rule_hits"]],
    on=["event_date", "event_id"],
    how="left",
)

label_ambiguity_penalty = np.where(event_scores["coarse_event_label"].astype("string").eq("market_commentary_or_misc"), 0.12, 0.0)
total_penalty = penalty_small_cluster + penalty_single_domain + penalty_duplicate + label_ambiguity_penalty

event_scores["impact_score"] = np.clip(event_scores["impact_score_raw"] - total_penalty, 0.0, 1.0)
event_scores["event_direction_score"] = event_scores["net_sentiment"].fillna(0.0)
event_scores["source_breadth"] = event_metrics["source_breadth"]

event_scores["event_confidence_score"] = np.clip(
    0.30 * event_scores["cohesion_score"].fillna(0.0) +
    0.22 * event_scores["source_breadth"].fillna(0.0) +
    0.18 * event_scores["theme_diversity"].fillna(0.0) +
    0.15 * event_scores["novelty_score"].fillna(0.0) +
    0.15 * event_scores["label_confidence_rule_based"].fillna(0.0) -
    total_penalty,
    0.0,
    1.0,
)

event_scores["confidence_bucket"] = event_scores["event_confidence_score"].map(confidence_bucket)
event_scores["warning_penalty"] = total_penalty

required_event_fields = [
    "event_id", "event_date", "n_rows", "n_unique_urls", "n_unique_domains", "domain_entropy",
    "n_unique_orgs", "n_unique_locs", "n_unique_themes", "avg_tone", "median_tone", "tone_std",
    "positive_share", "negative_share", "neutral_share", "net_sentiment", "source_breadth",
    "novelty_score", "impact_score", "event_direction_score", "event_confidence_score", "warning_penalty",
    "label_rule_hits",
]
assert_columns(event_scores, required_event_fields, "event_scores")

pretty_table(event_scores.sort_values("event_confidence_score", ascending=False), 12)

,event_date,event_id,n_rows,n_unique_urls,n_unique_domains,n_unique_orgs,n_unique_locs,n_unique_themes,avg_tone,median_tone,...,coarse_event_label,fine_event_label,label_confidence_rule_based,label_evidence_fields,label_rule_hits,impact_score,event_direction_score,event_confidence_score,confidence_bucket,warning_penalty
26055,2022-02-28 00:00:00+00:00,E20220228_a7ff5334,1032,1032,580,836,394,969,-3.723365,-3.450136,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""geopolitics_or_conflict"": [""conflict"", ""bord...",0.756407,-0.922481,0.502564,medium,0.0
35197,2022-07-12 00:00:00+00:00,E20220712_4d227ad7,1031,1031,618,534,355,926,-2.079462,-2.209945,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""geopolitics_or_conflict"": [""conflict"", ""bord...",0.743381,-0.623666,0.494846,medium,0.0
35628,2022-07-14 00:00:00+00:00,E20220714_5b6f014f,510,510,349,447,349,858,-1.957729,-2.027849,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.79,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""company_or_sector_news"": [""finance""], ""geopo...",0.661919,-0.705882,0.480254,medium,0.0
35509,2022-07-13 00:00:00+00:00,E20220713_bf9880b6,772,772,479,483,364,929,-1.948752,-1.785994,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.57,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""macro_economy_or_policy"": [""monetary""], ""com...",0.707755,-0.630829,0.471682,medium,0.0
41260,2022-09-26 00:00:00+00:00,E20220926_93da644a,892,892,567,435,319,811,-3.164290,-3.246753,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""macro_economy_or_policy"": [""government""], ""g...",0.736444,-0.865471,0.469191,medium,0.0
26185,2022-03-01 00:00:00+00:00,E20220301_f2792491,449,449,322,462,336,870,-2.397023,-2.419355,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""geopolitics_or_conflict"": [""conflict"", ""bord...",0.653186,-0.766147,0.461774,medium,0.0
28387,2022-03-31 00:00:00+00:00,E20220331_c503b619,519,519,372,389,321,843,-1.949252,-2.250804,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""geopolitics_or_conflict"": [""conflict"", ""bord...",0.662266,-0.753372,0.460368,medium,0.0
36311,2022-07-21 00:00:00+00:00,E20220721_3103ffa0,496,496,352,516,304,838,-1.661146,-1.708250,...,macro_economy_or_policy,rule_based::macro_economy_or_policy,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""macro_economy_or_policy"": [""rates"", ""governm...",0.659760,-0.558468,0.458409,medium,0.0
28463,2022-04-01 00:00:00+00:00,E20220401_b43399c5,569,569,372,406,310,822,-1.743957,-2.194169,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""geopolitics_or_conflict"": [""conflict"", ""bord...",0.667806,-0.787346,0.457474,medium,0.0
30127,2022-04-28 00:00:00+00:00,E20220428_0f653b22,511,511,357,572,346,826,-1.847793,-1.547988,...,geopolitics_or_conflict,rule_based::geopolitics_or_conflict,0.67,"top_theme_tokens,top_org_tokens,top_loc_tokens","{""macro_economy_or_policy"": [""government""], ""g...",0.660363,-0.765166,0.457142,medium,0.0


## 16) Daily Explanation Assembly


In [15]:
event_detail_export = event_scores.copy()
event_detail_export["event_date"] = pd.to_datetime(event_detail_export["event_date"], errors="coerce", utc=True).dt.floor("D")
event_detail_export["event_id"] = event_detail_export["event_id"].astype("string")

event_row_features = event_clusters_rows[[
    "event_date", "event_id", "timestamp_primary", "url", "domain_clean", "row_event_preview", "org_set", "theme_set", "loc_set",
]].copy()
event_row_features["event_date"] = pd.to_datetime(event_row_features["event_date"], errors="coerce", utc=True).dt.floor("D")
event_row_features["event_id"] = event_row_features["event_id"].astype("string")
event_profile = event_detail_export.set_index(["event_date", "event_id"])[["n_unique_domains"]].to_dict(orient="index")


def row_rep_score(r: pd.Series) -> float:
    key = (r["event_date"], r["event_id"])
    n_dom = float(event_profile.get(key, {}).get("n_unique_domains", 1.0))
    breadth_contrib = np.log1p(n_dom)
    recency = 0.0
    if pd.notna(r.get("timestamp_primary", pd.NaT)):
        recency = pd.to_datetime(r["timestamp_primary"], utc=True).hour / 24.0
    token_mass = float(len(r["org_set"]) + len(r["theme_set"]) + len(r["loc_set"]))
    return 0.45 * breadth_contrib + 0.25 * recency + 0.30 * np.log1p(token_mass)


def _clean_url_list(values) -> list[str]:
    if values is None:
        return []
    if isinstance(values, float) and np.isnan(values):
        return []
    if isinstance(values, (list, tuple, np.ndarray, pd.Series)):
        iterable = list(values)
    else:
        iterable = [values]
    out = []
    for value in iterable:
        if value is None:
            continue
        text = str(value).strip()
        if text and text != "<NA>":
            out.append(text)
    return out


def _first_nonempty_strings(series: pd.Series, limit: int = 5) -> list[str]:
    out = []
    seen = set()
    for value in series.tolist():
        if value is None or pd.isna(value):
            continue
        text = str(value).strip()
        if not text or text == "<NA>":
            continue
        if text not in seen:
            seen.add(text)
            out.append(text)
        if len(out) >= limit:
            break
    return out


event_row_features["rep_score"] = event_row_features.apply(row_rep_score, axis=1)
event_row_features = event_row_features.sort_values(["event_date", "event_id", "rep_score"], ascending=[True, True, False])
event_row_features = event_row_features.drop_duplicates(subset=["event_date", "event_id", "url"], keep="first")

repr_urls = event_row_features.groupby(["event_date", "event_id"], as_index=False).agg(
    representative_urls=("url", lambda s: _first_nonempty_strings(s, 5)),
    representative_domains=("domain_clean", lambda s: _first_nonempty_strings(s, 5)),
    representative_row_previews=("row_event_preview", lambda s: _first_nonempty_strings(s, 5)),
    top_orgs=("org_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:8]),
    top_themes=("theme_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:8]),
    top_locs=("loc_set", lambda s: sorted({x for arr in s if isinstance(arr, list) for x in arr})[:8]),
)

event_detail_export = event_detail_export.merge(repr_urls, on=["event_date", "event_id"], how="left")
event_detail_export["representative_urls"] = event_detail_export["representative_urls"].map(_clean_url_list)

auto_fallback_urls = event_clusters_rows.copy()
auto_fallback_urls["event_date"] = pd.to_datetime(auto_fallback_urls["event_date"], errors="coerce", utc=True).dt.floor("D")
auto_fallback_urls["event_id"] = auto_fallback_urls["event_id"].astype("string")
auto_fallback_urls = auto_fallback_urls.groupby(["event_date", "event_id"], as_index=False).agg(
    representative_urls_fallback=("url", lambda s: _first_nonempty_strings(s, 5)),
)
event_detail_export = event_detail_export.merge(auto_fallback_urls, on=["event_date", "event_id"], how="left")
event_detail_export["representative_urls"] = event_detail_export.apply(
    lambda r: _clean_url_list(r["representative_urls"]) if len(_clean_url_list(r["representative_urls"])) > 0 else _clean_url_list(r["representative_urls_fallback"]),
    axis=1,
)
event_detail_export = event_detail_export.drop(columns=["representative_urls_fallback"])


def event_warnings_from_row(r: pd.Series) -> list[str]:
    w = []
    if int(r.get("n_rows", 0)) < CFG.MIN_ROWS_PER_EVENT:
        w.append("tiny_cluster")
    if int(r.get("n_unique_domains", 0)) <= 1:
        w.append("single_domain_dominated")
    if float(r.get("label_confidence_rule_based", 0.0)) < 0.4:
        w.append("low_label_confidence")
    if float(r.get("novelty_score", 0.0)) < 0.2:
        w.append("low_novelty")
    if int(r.get("n_unique_urls", 0)) <= 1 and int(r.get("n_rows", 0)) >= 3:
        w.append("repeated_url_dominated")
    if int(r.get("n_unique_orgs", 0)) == 0 and int(r.get("n_unique_themes", 0)) == 0:
        w.append("sparse_entity_fields")
    if abs(float(r.get("event_direction_score", 0.0))) < 0.05:
        w.append("weak_sentiment_signal")
    if float(r.get("source_breadth", 0.0)) < 0.20:
        w.append("low_source_breadth")
    return w


event_detail_export["event_warnings"] = event_detail_export.apply(event_warnings_from_row, axis=1)
event_detail_export["mode"] = PRIMARY_PRODUCTION_MODE
event_detail_export["cluster_id"] = event_detail_export["event_id"]

# RELAXED publishable gate: condition on confidence OR size+breadth (not too strict with domain requirement).
event_detail_export["publishable"] = event_detail_export.apply(
    lambda r: bool(
        ((float(r.get("event_confidence_score", 0.0)) >= 0.40) and (int(r.get("n_rows", 0)) >= CFG.MIN_ROWS_PER_EVENT))
        or ((int(r.get("n_rows", 0)) >= 4) and (float(r.get("source_breadth", 0.0)) >= 0.35))
    )
    and ("sparse_entity_fields" not in r.get("event_warnings", [])),
    axis=1,
)


def build_one_line_summary(r: pd.Series) -> str:
    top_orgs = _clean_url_list(r["top_orgs"])
    top_locs = _clean_url_list(r["top_locs"])
    top_org = top_orgs[0] if top_orgs else "unknown_org"
    top_loc = top_locs[0] if top_locs else "unknown_loc"
    direction = "Positive" if r["event_direction_score"] > 0.05 else ("Negative" if r["event_direction_score"] < -0.05 else "Mixed")
    conf_level = confidence_bucket(float(r["event_confidence_score"]))
    return (
        f"{direction} {r['coarse_event_label']} event centered on {top_org} in {top_loc}, "
        f"supported by {int(r['n_rows'])} rows across {int(r['n_unique_domains'])} domains, "
        f"with {direction.lower()} tone and {conf_level} confidence."
    )


def build_rich_summary(r: pd.Series) -> str:
    fine = r["fine_event_label"] if pd.notna(r.get("fine_event_label", pd.NA)) else "na"
    warns = ", ".join(r["event_warnings"]) if isinstance(r["event_warnings"], list) and r["event_warnings"] else "none"
    return (
        f"mode={r['mode']} | label={r['coarse_event_label']} | fine={fine}; "
        f"orgs={_clean_url_list(r['top_orgs'])[:3]}; "
        f"themes={_clean_url_list(r['top_themes'])[:3]}; "
        f"locs={_clean_url_list(r['top_locs'])[:3]}; "
        f"source_breadth={float(r['source_breadth']):.3f}; impact={float(r['impact_score']):.3f}; novelty={float(r['novelty_score']):.3f}; "
        f"confidence={float(r['event_confidence_score']):.3f}; publishable={bool(r['publishable'])}; warnings={warns}."
    )

event_detail_export["event_summary_one_line"] = event_detail_export.apply(build_one_line_summary, axis=1)
event_detail_export["event_summary_rich"] = event_detail_export.apply(build_rich_summary, axis=1)
event_detail_export["short_explanation_text"] = event_detail_export["event_summary_one_line"]

daily_rows = []
daily_agent_payload = []
for d, sub in event_detail_export.groupby("event_date"):
    sub = sub.sort_values("event_confidence_score", ascending=False).copy()
    n_events = int(sub["event_id"].nunique())

    pos = sub[sub["event_direction_score"] > 0.05].head(CFG.MAX_EVENTS_PER_DAY_TO_EXPORT)
    neg = sub[sub["event_direction_score"] < -0.05].head(CFG.MAX_EVENTS_PER_DAY_TO_EXPORT)
    neu = sub[sub["event_direction_score"].abs() <= 0.05].head(CFG.MAX_EVENTS_PER_DAY_TO_EXPORT)

    dominant_labels = sub["coarse_event_label"].value_counts().head(3).index.tolist()
    dominant_themes = [x for xs in sub["top_themes"].dropna().tolist() if isinstance(xs, list) for x in xs][:20]
    dominant_orgs = [x for xs in sub["top_orgs"].dropna().tolist() if isinstance(xs, list) for x in xs][:20]
    dominant_locs = [x for xs in sub["top_locs"].dropna().tolist() if isinstance(xs, list) for x in xs][:20]

    overall_balance = float(sub["event_direction_score"].mean()) if len(sub) else 0.0
    overall_conf = float(sub["event_confidence_score"].mean()) if len(sub) else 0.0
    publishable_events = int(sub["publishable"].sum()) if len(sub) else 0
    degraded_events = int(len(sub) - publishable_events)

    daily_warnings = []
    if n_events == 0:
        daily_warnings.append("no_events")
    if (float((sub["n_unique_domains"] <= 1).mean()) > 0.75) if len(sub) else False:
        daily_warnings.append("single_domain_dominance")
    if (float((sub["event_confidence_score"] < 0.35).mean()) > 0.6) if len(sub) else False:
        daily_warnings.append("low_confidence_day")
    if publishable_events == 0 and n_events > 0:
        daily_warnings.append("no_publishable_events")

    publishable_day = bool(publishable_events > 0)

    daily_rows.append({
        "date": d,
        "mode": PRIMARY_PRODUCTION_MODE,
        "n_events": n_events,
        "n_publishable_events": publishable_events,
        "n_degraded_events": degraded_events,
        "publishable": publishable_day,
        "dominant_event_labels": json.dumps(dominant_labels),
        "dominant_orgs": json.dumps(dominant_orgs[:8]),
        "dominant_themes": json.dumps(dominant_themes[:8]),
        "dominant_locs": json.dumps(dominant_locs[:8]),
        "overall_event_balance": overall_balance,
        "overall_confidence_score": overall_conf,
        "warnings": json.dumps(daily_warnings),
    })

    def pack_events(df_sub: pd.DataFrame) -> list[dict[str, Any]]:
        out = []
        for _, r in df_sub.iterrows():
            out.append({
                "date": str(pd.to_datetime(r["event_date"], utc=True).date()),
                "mode": r["mode"],
                "event_id": r["event_id"],
                "cluster_id": r["cluster_id"],
                "label": r["coarse_event_label"],
                "confidence": float(r["event_confidence_score"]),
                "impact_score": float(r["impact_score"]),
                "novelty_score": float(r["novelty_score"]),
                "source_breadth": float(r["source_breadth"]),
                "top_orgs": _clean_url_list(r["top_orgs"]),
                "top_locs": _clean_url_list(r["top_locs"]),
                "top_themes": _clean_url_list(r["top_themes"]),
                "representative_urls": _clean_url_list(r["representative_urls"]),
                "warning_flags": r["event_warnings"] if isinstance(r["event_warnings"], list) else [],
                "publishable": bool(r["publishable"]),
                "summary": r["event_summary_one_line"],
            })
        return out

    daily_agent_payload.append({
        "date": str(d.date()),
        "mode": PRIMARY_PRODUCTION_MODE,
        "publishable": publishable_day,
        "n_events": n_events,
        "n_publishable_events": publishable_events,
        "n_degraded_events": degraded_events,
        "overall_event_balance": overall_balance,
        "overall_confidence_score": overall_conf,
        "warnings": daily_warnings,
        "top_positive_events": pack_events(pos),
        "top_negative_events": pack_events(neg),
        "top_neutral_or_high_importance_events": pack_events(neu),
    })

daily_event_explanations = pd.DataFrame(daily_rows).sort_values("date")

event_warnings_export = event_detail_export[["event_date", "event_id", "event_warnings"]].copy()
event_warnings_export["event_warnings"] = event_warnings_export["event_warnings"].map(json.dumps)
daily_warnings = daily_event_explanations[["date", "warnings"]].copy()
save_csv(event_warnings_export, CFG.OUTPUT_DIR / "artifacts" / "event_warnings.csv")
save_csv(daily_warnings, CFG.OUTPUT_DIR / "artifacts" / "daily_warnings.csv")

rep_url_coverage = float(event_detail_export["representative_urls"].map(lambda x: isinstance(x, list) and len(x) > 0).mean()) if len(event_detail_export) else 0.0
print(f"Daily explanation assembly complete. Representative URL coverage={rep_url_coverage:.4f}")
pretty_table(daily_event_explanations, 10)

Daily explanation assembly complete. Representative URL coverage=1.0000


,date,mode,n_events,n_publishable_events,n_degraded_events,publishable,dominant_event_labels,dominant_orgs,dominant_themes,dominant_locs,overall_event_balance,overall_confidence_score,warnings
0,2021-01-01 00:00:00+00:00,GKG_structured_mode,39,1,38,True,"[""macro_economy_or_policy"", ""legal_or_regulato...","[""aegea biotechnologies inc"", ""b xinhua news a...","[""affect"", ""alliance"", ""armedconflict"", ""ban"",...","[""america"", ""american"", ""americans"", ""argentin...",-0.418543,0.129040,"[""low_confidence_day""]"
1,2021-01-02 00:00:00+00:00,GKG_structured_mode,26,1,25,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""airport international"", ""alamy"", ""associated...","[""affect"", ""agriculture"", ""appointment"", ""arme...","[""algeria"", ""algerian"", ""america"", ""american"",...",-0.092339,0.117746,"[""low_confidence_day""]"
2,2021-01-03 00:00:00+00:00,GKG_structured_mode,32,1,31,True,"[""macro_economy_or_policy"", ""geopolitics_or_co...","[""association midwest economy"", ""body center s...","[""affect"", ""agriculture"", ""alliance"", ""appoint...","[""algeria"", ""algerian"", ""america"", ""american"",...",-0.147420,0.099321,"[""low_confidence_day""]"
3,2021-01-04 00:00:00+00:00,GKG_structured_mode,105,1,104,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""africa finance corporation"", ""airbus"", ""al b...","[""act_makestatement"", ""affect"", ""agriculture"",...","[""albania"", ""america"", ""american"", ""argentina""...",-0.333958,0.048267,"[""single_domain_dominance"", ""low_confidence_day""]"
4,2021-01-05 00:00:00+00:00,GKG_structured_mode,86,3,83,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""al bank"", ""al senate country"", ""alphabet inc...","[""affect"", ""agriculture"", ""armedconflict"", ""ar...","[""algeria"", ""amber hill, lincolnshire, united ...",-0.325805,0.076383,"[""low_confidence_day""]"
5,2021-01-06 00:00:00+00:00,GKG_structured_mode,97,2,95,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""a exchange"", ""administration of joe biden"", ...","[""act_makestatement"", ""affect"", ""agriculture"",...","[""algeria"", ""algerian"", ""america"", ""american"",...",-0.085285,0.075821,"[""low_confidence_day""]"
6,2021-01-07 00:00:00+00:00,GKG_structured_mode,93,1,92,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""administration of foreign exchange"", ""admini...","[""affect"", ""agriculture"", ""alliance"", ""armedco...","[""america"", ""american"", ""americans"", ""arizona,...",-0.263809,0.093887,"[""low_confidence_day""]"
7,2021-01-08 00:00:00+00:00,GKG_structured_mode,89,1,88,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""a congress"", ""adidas"", ""aegea biotechnologie...","[""affect"", ""agriculture"", ""appointment"", ""arme...","[""america"", ""american"", ""americans"", ""argentin...",-0.070314,0.070272,"[""low_confidence_day""]"
8,2021-01-09 00:00:00+00:00,GKG_structured_mode,29,1,28,True,"[""macro_economy_or_policy"", ""company_or_sector...","[""a court"", ""a whole deutsche bank"", ""aegea bi...","[""agriculture"", ""armedconflict"", ""arrest"", ""cl...","[""abu dhabi, abu z\u00b8aby, united arab emira...",-0.015152,0.117710,"[""low_confidence_day""]"
9,2021-01-10 00:00:00+00:00,GKG_structured_mode,37,2,35,True,"[""macro_economy_or_policy"", ""geopolitics_or_co...","[""a central bank"", ""bank great wall"", ""bank is...","[""act_makestatement"", ""affect"", ""agriculture"",...","[""abu dhabi, abu z\u00b8aby, united arab emira...",-0.430901,0.093331,"[""low_confidence_day""]"


In [16]:
# Day-level publishability override (strict and auditable).
# This does not redesign clustering; it only tightens day packet quality.

day_rule_rows = []
for d, sub in event_detail_export.groupby("event_date"):
    sub = sub.sort_values(["event_confidence_score", "impact_score"], ascending=[False, False]).copy()
    top_n = min(5, len(sub))
    top_sub = sub.head(top_n).copy()

    n_events = int(len(sub))
    n_publishable = int(sub["publishable"].sum())
    top_publishable_share = float(top_sub["publishable"].mean()) if top_n else 0.0
    top_rep_url_coverage = float(top_sub["representative_urls"].map(lambda x: isinstance(x, list) and len(x) > 0).mean()) if top_n else 0.0
    top_mean_domains = float(pd.to_numeric(top_sub["n_unique_domains"], errors="coerce").fillna(0).mean()) if top_n else 0.0
    day_conf = float(pd.to_numeric(sub["event_confidence_score"], errors="coerce").fillna(0).mean()) if n_events else 0.0

    rule_ge2_publishable = n_publishable >= 2
    rule_quality_mix = (
        (n_publishable >= 1)
        and (top_publishable_share >= 0.40)
        and (top_rep_url_coverage >= 0.80)
        and (top_mean_domains >= 2.0)
        and (day_conf >= 0.28)
    )
    publishable_day_strict = bool(rule_ge2_publishable or rule_quality_mix)

    day_rule_rows.append({
        "date": d,
        "n_events": n_events,
        "n_publishable_events": n_publishable,
        "top_publishable_share": top_publishable_share,
        "top_rep_url_coverage": top_rep_url_coverage,
        "top_mean_domains": top_mean_domains,
        "day_confidence_mean": day_conf,
        "rule_ge2_publishable": int(rule_ge2_publishable),
        "rule_quality_mix": int(rule_quality_mix),
        "publishable_strict": int(publishable_day_strict),
    })

day_rule_audit = pd.DataFrame(day_rule_rows).sort_values("date")

daily_event_explanations = daily_event_explanations.merge(
    day_rule_audit[["date", "publishable_strict", "top_publishable_share", "top_rep_url_coverage", "top_mean_domains", "day_confidence_mean", "rule_ge2_publishable", "rule_quality_mix"]],
    on="date",
    how="left",
)
daily_event_explanations["publishable"] = daily_event_explanations["publishable_strict"].fillna(0).astype(int).astype(bool)
daily_event_explanations = daily_event_explanations.drop(columns=["publishable_strict"])

daily_event_explanations["warnings"] = daily_event_explanations.apply(
    lambda r: (
        json.dumps(sorted(list(set((json.loads(r["warnings"]) if isinstance(r.get("warnings"), str) else []) + (["strict_day_gate_fail"] if not bool(r.get("publishable", False)) else [])))))
        if isinstance(r.get("warnings"), str)
        else json.dumps(["strict_day_gate_fail"] if not bool(r.get("publishable", False)) else [])
    ),
    axis=1,
)

for payload in daily_agent_payload:
    d = pd.to_datetime(payload["date"], utc=True).floor("D")
    row = day_rule_audit[day_rule_audit["date"] == d]
    if len(row):
        payload["publishable"] = bool(int(row.iloc[0]["publishable_strict"]))
        payload["day_publishability_audit"] = {
            "n_publishable_events": int(row.iloc[0]["n_publishable_events"]),
            "top_publishable_share": float(row.iloc[0]["top_publishable_share"]),
            "top_rep_url_coverage": float(row.iloc[0]["top_rep_url_coverage"]),
            "top_mean_domains": float(row.iloc[0]["top_mean_domains"]),
            "day_confidence_mean": float(row.iloc[0]["day_confidence_mean"]),
            "rule_ge2_publishable": int(row.iloc[0]["rule_ge2_publishable"]),
            "rule_quality_mix": int(row.iloc[0]["rule_quality_mix"]),
        }

save_csv(day_rule_audit, CFG.OUTPUT_DIR / "artifacts" / "day_publishability_rule_audit.csv")
print("Strict day-level publishability gate applied.")
print(day_rule_audit[["publishable_strict", "rule_ge2_publishable", "rule_quality_mix"]].sum().to_string())

Strict day-level publishability gate applied.
publishable_strict      1032
rule_ge2_publishable    1032
rule_quality_mix           0


## 17) Optional Link to Forecast Outputs


In [17]:
forecast_df = loaded_cache.get("daily_predictions")

if isinstance(forecast_df, pd.DataFrame) and len(forecast_df) > 0 and "date" in forecast_df.columns:
    fc = forecast_df.copy()
    fc["date"] = pd.to_datetime(fc["date"], errors="coerce", utc=True).dt.floor("D")

    cols = [c for c in ["date", "y_prob", "y_pred", "y_true", "model"] if c in fc.columns]
    fc = fc[cols].copy()

    daily_event_explanations = daily_event_explanations.merge(fc, on="date", how="left")
    daily_event_explanations.rename(columns={"y_prob": "forecast_prob", "y_pred": "forecast_pred", "y_true": "forecast_realized"}, inplace=True)

    daily_event_explanations["event_direction_sign"] = np.sign(daily_event_explanations["overall_event_balance"].fillna(0.0))
    daily_event_explanations["forecast_direction_sign"] = np.where(
        daily_event_explanations["forecast_pred"].isna(),
        np.nan,
        np.where(daily_event_explanations["forecast_pred"] > 0, 1, -1),
    )
    daily_event_explanations["explanation_alignment_flag"] = np.where(
        daily_event_explanations["forecast_direction_sign"].isna(),
        pd.NA,
        np.where(daily_event_explanations["event_direction_sign"] == daily_event_explanations["forecast_direction_sign"], 1, 0),
    )

    daily_event_explanations["explanation_alignment_score"] = (
        1.0 - (daily_event_explanations["event_direction_sign"] - daily_event_explanations["forecast_direction_sign"]).abs().fillna(1.0) / 2.0
    )

    def disagreement_reason(r: pd.Series) -> str:
        if pd.isna(r.get("forecast_direction_sign", np.nan)):
            return "no_forecast"
        if r["explanation_alignment_flag"] == 1:
            return "aligned"
        if abs(float(r.get("overall_event_balance", 0.0))) < 0.15:
            return "weak_event_direction"
        if "single_domain_dominance" in str(r.get("warnings", "")):
            return "concentrated_coverage"
        return "forecast_event_disagreement"

    daily_event_explanations["disagreement_reason_heuristic"] = daily_event_explanations.apply(disagreement_reason, axis=1)

    strong_negative_vs_forecast_pos = daily_event_explanations[
        (daily_event_explanations["overall_event_balance"] < -0.35) & (daily_event_explanations["forecast_direction_sign"] == 1)
    ].copy()
    strong_positive_vs_forecast_neg = daily_event_explanations[
        (daily_event_explanations["overall_event_balance"] > 0.35) & (daily_event_explanations["forecast_direction_sign"] == -1)
    ].copy()

    save_csv(strong_negative_vs_forecast_pos, CFG.OUTPUT_DIR / "artifacts" / "strong_negative_events_vs_positive_forecast.csv")
    save_csv(strong_positive_vs_forecast_neg, CFG.OUTPUT_DIR / "artifacts" / "strong_positive_events_vs_negative_forecast.csv")

    fc_map = daily_event_explanations.set_index(daily_event_explanations["date"].astype(str))[
        ["forecast_prob", "forecast_pred", "forecast_realized", "explanation_alignment_flag", "explanation_alignment_score", "disagreement_reason_heuristic"]
    ].to_dict(orient="index")
    for item in daily_agent_payload:
        k = str(pd.to_datetime(item["date"], utc=True).floor("D"))
        if k in fc_map:
            item["forecast_context"] = {
                "forecast_prob": fc_map[k].get("forecast_prob"),
                "forecast_pred": fc_map[k].get("forecast_pred"),
                "forecast_realized": fc_map[k].get("forecast_realized"),
                "alignment_flag": fc_map[k].get("explanation_alignment_flag"),
                "alignment_score": fc_map[k].get("explanation_alignment_score"),
                "disagreement_reason": fc_map[k].get("disagreement_reason_heuristic"),
            }

    print("Forecast linkage completed.")
else:
    print("Forecast outputs unavailable; continuing with event-only summaries.")


Forecast linkage completed.


## 18) Quality Checks / Sanity Checks


In [18]:
warnings_rows = []
check_rows = []

def add_check(name: str, status: str, detail: str) -> None:
    check_rows.append({"check": name, "status": status, "detail": detail})

dup_event_ids = event_scores.duplicated(subset=["event_date", "event_id"]).sum()
add_check("duplicate_event_ids", "pass" if dup_event_ids == 0 else "fail", f"count={int(dup_event_ids)}")

leak_rows = int((event_clusters_rows["event_date"] != pd.to_datetime(event_clusters_rows["date"], utc=True).dt.floor("D")).sum())
add_check("future_leakage", "pass" if leak_rows == 0 else "fail", f"count={int(leak_rows)}")

share_sum = event_scores[["positive_share", "negative_share", "neutral_share"]].sum(axis=1)
bad_share_rows = int((share_sum - 1.0).abs().gt(0.05).sum())
add_check("sentiment_share_sum", "pass" if bad_share_rows == 0 else "warn", f"count={int(bad_share_rows)}")

singleton_rate_after = float((event_scores["n_rows"] == 1).mean()) if len(event_scores) else np.nan
singleton_rate_before = float((cand_sizes["n_rows"] == 1).mean()) if "cand_sizes" in globals() and len(cand_sizes) else np.nan
if pd.notna(singleton_rate_after) and singleton_rate_after > 0.75:
    warnings_rows.append({"warning_type": "singleton_explosion", "count": int((event_scores["n_rows"] == 1).sum()), "detail": f"rate={singleton_rate_after:.4f}"})
add_check("singleton_explosion", "warn" if (pd.notna(singleton_rate_after) and singleton_rate_after > 0.75) else "pass", f"before={singleton_rate_before:.4f}, after={singleton_rate_after:.4f}")

label_dist = event_scores["coarse_event_label"].value_counts(dropna=False)
label_collapse = bool(len(label_dist) > 0 and float(label_dist.max() / max(1, label_dist.sum())) > 0.80)
if label_collapse:
    warnings_rows.append({"warning_type": "label_collapse", "count": int(label_dist.max()), "detail": "one label dominates >80%"})
add_check("label_collapse", "warn" if label_collapse else "pass", f"top_share={float(label_dist.max()/max(1,label_dist.sum())) if len(label_dist) else np.nan:.4f}")

low_breadth_rate = float((event_scores["n_unique_domains"] <= 1).mean()) if len(event_scores) else np.nan
if pd.notna(low_breadth_rate) and low_breadth_rate > 0.50:
    warnings_rows.append({"warning_type": "low_source_breadth", "count": int((event_scores["n_unique_domains"] <= 1).sum()), "detail": f"rate={low_breadth_rate:.4f}"})
add_check("source_breadth", "warn" if (pd.notna(low_breadth_rate) and low_breadth_rate > 0.50) else "pass", f"low_breadth_rate={low_breadth_rate:.4f}")

weak_rep_rate = float(event_detail_export["representative_urls"].map(lambda x: not isinstance(x, list) or len(x) == 0).mean()) if len(event_detail_export) else np.nan
add_check("representative_evidence", "warn" if (pd.notna(weak_rep_rate) and weak_rep_rate > 0.20) else "pass", f"empty_rep_url_rate={weak_rep_rate:.4f}")

concentration_rate = float((event_scores["domain_top_share"].fillna(1.0) > 0.80).mean()) if "domain_top_share" in event_scores.columns else np.nan
add_check("domain_concentration", "warn" if (pd.notna(concentration_rate) and concentration_rate > 0.40) else "pass", f"dominant_domain_rate={concentration_rate:.4f}")

publishable_events = int(event_detail_export["publishable"].sum()) if "publishable" in event_detail_export.columns else 0
degraded_events = int(len(event_detail_export) - publishable_events)
add_check("publishable_events", "pass" if publishable_events > 0 else "warn", f"publishable={publishable_events}, degraded={degraded_events}")

publishable_days = int(daily_event_explanations["publishable"].sum()) if "publishable" in daily_event_explanations.columns else 0
degraded_days = int(len(daily_event_explanations) - publishable_days)
add_check("publishable_daily_packets", "pass" if publishable_days > 0 else "warn", f"publishable_days={publishable_days}, degraded_days={degraded_days}")

non_empty = daily_event_explanations[daily_event_explanations["n_events"] > 0].copy()
if len(non_empty):
    median_events_non_empty_day = float(non_empty["n_events"].median())
    pct_days_ge_2_events = float((non_empty["n_events"] >= 2).mean())
    pct_days_ge_3_events = float((non_empty["n_events"] >= 3).mean())
else:
    median_events_non_empty_day = 0.0
    pct_days_ge_2_events = 0.0
    pct_days_ge_3_events = 0.0

multi_event_sanity = pd.DataFrame([
    {"metric": "median_events_per_non_empty_day", "value": median_events_non_empty_day, "threshold": CFG.MIN_MEDIAN_EVENTS_PER_NON_EMPTY_DAY},
    {"metric": "pct_days_ge_2_events", "value": pct_days_ge_2_events, "threshold": CFG.MIN_PCT_DAYS_GE2_EVENTS},
    {"metric": "pct_days_ge_3_events", "value": pct_days_ge_3_events, "threshold": CFG.MIN_PCT_DAYS_GE3_EVENTS},
])
save_csv(multi_event_sanity, CFG.OUTPUT_DIR / "artifacts" / "multi_event_sanity.csv")

collapse_risk = (
    (median_events_non_empty_day < CFG.MIN_MEDIAN_EVENTS_PER_NON_EMPTY_DAY) or
    (pct_days_ge_2_events < CFG.MIN_PCT_DAYS_GE2_EVENTS) or
    (pct_days_ge_3_events < CFG.MIN_PCT_DAYS_GE3_EVENTS)
)
add_check(
    "multi_event_sanity",
    "warn" if collapse_risk else "pass",
    f"median={median_events_non_empty_day:.2f}, pct_ge2={pct_days_ge_2_events:.3f}, pct_ge3={pct_days_ge_3_events:.3f}",
)

before_after_metrics = pd.DataFrame([
    {"metric": "singleton_rate_before_candidates", "value": singleton_rate_before},
    {"metric": "singleton_rate_after_clusters", "value": singleton_rate_after},
    {"metric": "source_breadth_before_candidates_proxy", "value": float(cand_sizes['n_rows'].gt(1).mean()) if 'cand_sizes' in globals() and len(cand_sizes) else np.nan},
    {"metric": "source_breadth_after_events", "value": float(event_scores['n_unique_domains'].mean()) if len(event_scores) else np.nan},
    {"metric": "publishable_daily_packets", "value": publishable_days},
    {"metric": "degraded_daily_packets", "value": degraded_days},
])

label_distribution_after = event_scores["coarse_event_label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count")

event_quality_warnings = pd.DataFrame(warnings_rows)
quality_check_table = pd.DataFrame(check_rows)

save_csv(event_quality_warnings, CFG.OUTPUT_DIR / "artifacts" / "event_quality_warnings.csv")
save_csv(quality_check_table, CFG.OUTPUT_DIR / "artifacts" / "quality_check_table.csv")
save_csv(before_after_metrics, CFG.OUTPUT_DIR / "artifacts" / "before_after_quality_metrics.csv")
save_csv(label_distribution_after, CFG.OUTPUT_DIR / "artifacts" / "label_distribution_after.csv")

print("Quality checks complete.")
pretty_table(quality_check_table, 40)

Quality checks complete.


,check,status,detail
0,duplicate_event_ids,pass,count=0
1,future_leakage,pass,count=0
2,sentiment_share_sum,pass,count=0
3,singleton_explosion,pass,"before=0.7218, after=0.6515"
4,label_collapse,pass,top_share=0.7298
5,source_breadth,warn,low_breadth_rate=0.6917
6,representative_evidence,pass,empty_rep_url_rate=0.0000
7,domain_concentration,warn,dominant_domain_rate=0.6921
8,publishable_events,pass,"publishable=3204, degraded=125811"
9,publishable_daily_packets,pass,"publishable_days=1032, degraded_days=777"


## 19) Example Explanations


In [19]:
disp = daily_event_explanations.copy()
disp = disp.sort_values("date")

rank_col = "overall_confidence_score"
pos_days = disp.sort_values(["overall_event_balance", rank_col], ascending=[False, False]).head(3)
neg_days = disp.sort_values(["overall_event_balance", rank_col], ascending=[True, False]).head(3)
mixed_days = disp.assign(abs_bal=disp["overall_event_balance"].abs()).sort_values(["abs_bal", rank_col], ascending=[True, False]).head(3)
low_conf_days = disp.sort_values("overall_confidence_score", ascending=True).head(3)

example_days = pd.concat([
    pos_days.assign(example_group="strongest_positive"),
    neg_days.assign(example_group="strongest_negative"),
    mixed_days.assign(example_group="mixed_signal"),
    low_conf_days.assign(example_group="low_confidence"),
]).drop_duplicates(subset=["date", "example_group"]).sort_values(["example_group", "date"])


def unpack_json_list(x: Any) -> list[str]:
    if isinstance(x, list):
        return [str(v) for v in x]
    if isinstance(x, str):
        try:
            v = json.loads(x)
            if isinstance(v, list):
                return [str(t) for t in v]
        except Exception:
            return []
    return []


display_rows = []
for _, drow in example_days.iterrows():
    d = pd.to_datetime(drow["date"], utc=True).floor("D")
    top_events = event_detail_export[event_detail_export["event_date"] == d].sort_values(
        ["event_confidence_score", "impact_score"], ascending=[False, False]
    ).head(3)

    top_event_summaries = top_events["event_summary_one_line"].astype("string").tolist()
    top_event_urls = []
    for urls in top_events["representative_urls"].tolist():
        if isinstance(urls, list):
            top_event_urls.extend(urls[:2])

    display_rows.append({
        "example_group": drow["example_group"],
        "date": d,
        "publishable": bool(drow.get("publishable", False)),
        "forecast_prob": drow.get("forecast_prob", np.nan),
        "forecast_pred": drow.get("forecast_pred", np.nan),
        "forecast_realized": drow.get("forecast_realized", np.nan),
        "overall_event_balance": drow.get("overall_event_balance", np.nan),
        "n_events": drow.get("n_events", np.nan),
        "n_publishable_events": drow.get("n_publishable_events", np.nan),
        "overall_confidence_score": drow.get("overall_confidence_score", np.nan),
        "top_labels": drow.get("dominant_event_labels", "[]"),
        "top_orgs": drow.get("dominant_orgs", "[]"),
        "top_themes": drow.get("dominant_themes", "[]"),
        "top_locs": drow.get("dominant_locs", "[]"),
        "warnings": drow.get("warnings", "[]"),
        "top_event_summaries": json.dumps(top_event_summaries[:3]),
        "representative_urls": json.dumps(list(dict.fromkeys(top_event_urls))[:6]),
    })

example_display = pd.DataFrame(display_rows)
save_csv(example_display, CFG.OUTPUT_DIR / "artifacts" / "example_explanations_showcase.csv")
pretty_table(example_display, 20)

,example_group,date,publishable,forecast_prob,forecast_pred,forecast_realized,overall_event_balance,n_events,n_publishable_events,overall_confidence_score,top_labels,top_orgs,top_themes,top_locs,warnings,top_event_summaries,representative_urls
0,low_confidence,2021-04-17 00:00:00+00:00,False,NaN,NaN,NaN,-0.452242,27,1,0.011292,"[""macro_economy_or_policy"", ""company_or_sector...","[""a central bank to venezuela"", ""administratio...","[""act_makestatement"", ""affect"", ""agriculture"",...","[""abu dhabi, abu z\u00b8aby, united arab emira...","[""low_confidence_day"", ""single_domain_dominanc...","[""Negative macro_economy_or_policy event cente...","[""http://www.econotimes.com/Americas-Roundup-G..."
1,low_confidence,2021-08-14 00:00:00+00:00,False,NaN,NaN,NaN,-0.509286,28,1,0.018135,"[""macro_economy_or_policy"", ""company_or_sector...","[""accenture"", ""administrative department"", ""am...","[""affect"", ""agriculture"", ""alliance"", ""armedco...","[""afghan"", ""algeria"", ""algerian"", ""america"", ""...","[""low_confidence_day"", ""single_domain_dominanc...","[""Negative macro_economy_or_policy event cente...","[""https://www.calcuttanews.net/news/270693008/..."
2,low_confidence,2022-11-11 00:00:00+00:00,False,NaN,NaN,NaN,-0.228444,15,1,0.019124,"[""macro_economy_or_policy"", ""geopolitics_or_co...","[""airlines argentina"", ""analysis economic in g...","[""affect"", ""agriculture"", ""alliance"", ""armedco...","[""american"", ""americans"", ""argentina"", ""aussie...","[""low_confidence_day"", ""single_domain_dominanc...","[""Positive macro_economy_or_policy event cente...","[""https://utro.ru/news/ukraine/2022/11/11/1519..."
3,mixed_signal,2021-09-06 00:00:00+00:00,True,NaN,NaN,NaN,0.000632,56,2,0.083107,"[""macro_economy_or_policy"", ""company_or_sector...","[""a central bank to venezuela"", ""access bank"",...","[""affect"", ""agriculture"", ""aid_humanitarian"", ...","[""al bank al ahli, al jizah, egypt"", ""alexandr...","[""low_confidence_day""]","[""Negative geopolitics_or_conflict event cente...","[""https://www.cityindex.co.uk/market-analysis/..."
4,mixed_signal,2022-05-30 00:00:00+00:00,False,NaN,NaN,NaN,-0.000096,74,1,0.090119,"[""macro_economy_or_policy"", ""company_or_sector...","[""a central bank"", ""a central bank to venezuel...","[""affect"", ""agriculture"", ""alliance"", ""appoint...","[""america"", ""american"", ""americans"", ""amsterda...","[""low_confidence_day"", ""strict_day_gate_fail""]","[""Positive geopolitics_or_conflict event cente...","[""http://www.cfi.net.cn/p20220530000689.html"",..."
5,mixed_signal,2024-08-30 00:00:00+00:00,False,NaN,NaN,NaN,-0.001424,93,1,0.069667,"[""macro_economy_or_policy"", ""company_or_sector...","[""a bank"", ""a bottom monetary international"", ...","[""affect"", ""agriculture"", ""alliance"", ""appoint...","[""al masraf al markazi, dimashq, syria"", ""alex...","[""low_confidence_day"", ""strict_day_gate_fail""]","[""Negative geopolitics_or_conflict event cente...","[""https://www.philippinetimes.com/news/2745400..."
6,strongest_negative,2022-03-27 00:00:00+00:00,True,NaN,NaN,NaN,-0.690303,16,2,0.054760,"[""macro_economy_or_policy"", ""legal_or_regulato...","[""a central bank"", ""alliance europe europe"", ""...","[""act_makestatement"", ""affect"", ""agriculture"",...","[""afghanistan"", ""akure, ondo, nigeria"", ""alger...","[""low_confidence_day"", ""single_domain_dominance""]","[""Negative geopolitics_or_conflict event cente...","[""https://www.hellenicshippingnews.com/challen..."
7,strongest_negative,2022-06-19 00:00:00+00:00,False,NaN,NaN,NaN,-0.691525,27,1,0.098514,"[""macro_economy_or_policy"", ""geopolitics_or_co...","[""a central bank"", ""al bank al ahli"", ""al bank...","[""affect"", ""agriculture"", ""appointment"", ""arme...","[""al bank al ahli, al jizah, egypt"", ""america""...","[""low_confidence_day"", ""strict_day_gate_fail""]","[""Negative geopolitics_or_conflict event cente...","[""https://finance.eastmoney.com/a/202206202417..."
8,stronge

## Event Diagnostics Plots (Quality + Stability)`nQuick plots to detect instability: event volume, publishable-rate drift, cluster-size distribution, and most common warning flags.`n

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _ensure_dir(p: Path) -> None:
    try:
        safe_mkdir(p)
    except Exception:
        pass

out_dir = CFG.OUTPUT_DIR / "artifacts" / "plots"
_ensure_dir(out_dir)

# Daily volume + publishable rate
if "daily_event_explanations" in globals() and isinstance(daily_event_explanations, pd.DataFrame) and not daily_event_explanations.empty:
    d = daily_event_explanations.copy()
    d["date"] = pd.to_datetime(d["date"], errors="coerce", utc=True)
    d = d.dropna(subset=["date"]).sort_values("date")

    plt.figure(figsize=(12, 4))
    if "n_events" in d.columns:
        plt.plot(d["date"], pd.to_numeric(d["n_events"], errors="coerce"), label="n_events", linewidth=1.4)
    if "n_publishable_events" in d.columns:
        plt.plot(d["date"], pd.to_numeric(d["n_publishable_events"], errors="coerce"), label="n_publishable_events", linewidth=1.4)
    plt.title("Daily Event Volume")
    plt.xlabel("Date")
    plt.ylabel("Count")
    plt.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(out_dir / "event_daily_volume.png", dpi=160)
    plt.show()

    if "publishable" in d.columns:
        plt.figure(figsize=(12, 4))
        pub = pd.to_numeric(d["publishable"], errors="coerce").fillna(0.0)
        plt.plot(d["date"], pub.rolling(14, min_periods=3).mean(), label="publishable_rate_14d", linewidth=1.6)
        plt.ylim(-0.05, 1.05)
        plt.title("Publishable Day Rate (14-day rolling)")
        plt.xlabel("Date")
        plt.ylabel("Rate")
        plt.legend(fontsize=9)
        plt.tight_layout()
        plt.savefig(out_dir / "event_publishable_rate.png", dpi=160)
        plt.show()
else:
    print("Plot skip: daily_event_explanations missing/empty")

# Cluster size distribution
if "event_scores" in globals() and isinstance(event_scores, pd.DataFrame) and not event_scores.empty and "n_rows" in event_scores.columns:
    plt.figure(figsize=(7, 4))
    sizes = pd.to_numeric(event_scores["n_rows"], errors="coerce").dropna().clip(upper=50)
    plt.hist(sizes, bins=25)
    plt.title("Cluster Size Distribution (clipped at 50)")
    plt.xlabel("n_rows")
    plt.ylabel("count")
    plt.tight_layout()
    plt.savefig(out_dir / "event_cluster_size_hist.png", dpi=160)
    plt.show()
else:
    print("Plot skip: event_scores missing/empty")

# Warning frequency
if "event_detail_export" in globals() and isinstance(event_detail_export, pd.DataFrame) and not event_detail_export.empty and "event_warnings" in event_detail_export.columns:
    tmp = event_detail_export.copy()

    def _as_list(x):
        if isinstance(x, list):
            return x
        if isinstance(x, str):
            try:
                v = json.loads(x)
                return v if isinstance(v, list) else []
            except Exception:
                return []
        return []

    warns = tmp["event_warnings"].map(_as_list)
    flat = pd.Series([w for arr in warns.tolist() for w in arr if isinstance(w, str)])
    top = flat.value_counts().head(12)
    if len(top):
        plt.figure(figsize=(10, 4))
        top.sort_values().plot(kind="barh")
        plt.title("Top Event Warning Flags")
        plt.xlabel("count")
        plt.tight_layout()
        plt.savefig(out_dir / "event_warning_flags_top.png", dpi=160)
        plt.show()
else:
    print("Plot skip: event_detail_export missing/empty")


## 20) Export Artifacts


In [ ]:
if globals().get("SURROGATE_MODE_ACTIVE", False):
    raise RuntimeError("Surrogate mode is disabled for official exports. Clear the surrogate/debug path before exporting production artifacts.")

if str(globals().get("PRIMARY_PRODUCTION_MODE", "")).strip() == "degraded_debug_mode":
    raise RuntimeError(
        "Official exports are blocked in degraded_debug_mode. Resolve required structured inputs and rerun in GKG_structured_mode before exporting."
    )

required_objects = {
    "schema_audit": globals().get("schema_audit", None),
    "row_event_candidates": globals().get("row_event_candidates", None),
    "event_clusters_rows": globals().get("event_clusters_rows", None),
    "event_clusters_summary": globals().get("event_clusters_summary", None),
    "label_coverage": globals().get("label_coverage", None),
    "daily_event_explanations": globals().get("daily_event_explanations", None),
    "event_detail_export": globals().get("event_detail_export", None),
    "event_quality_warnings": globals().get("event_quality_warnings", None),
}
missing_required_objs = [k for k, v in required_objects.items() if not isinstance(v, pd.DataFrame) or len(v) == 0]
if missing_required_objs:
    raise RuntimeError(
        f"Official export blocked: required tables missing or empty: {', '.join(missing_required_objs)}"
    )

if not isinstance(globals().get("daily_agent_payload", None), list) or len(daily_agent_payload) == 0:
    raise RuntimeError("Official export blocked: daily_agent_payload is missing or empty.")

art = CFG.OUTPUT_DIR / "artifacts"
plots_dir = art / "plots"
reports_dir = art / "reports"

# Required core exports.
save_csv(schema_audit, art / "event_schema_audit.csv")
save_parquet(row_event_candidates, art / "row_event_candidates.parquet")
save_parquet(event_clusters_rows, art / "event_clusters_rows.parquet")
save_csv(event_clusters_summary, art / "event_clusters_summary.csv")
save_csv(label_coverage, art / "event_label_coverage.csv")
save_csv(daily_event_explanations, art / "daily_event_explanations.csv")
save_json(daily_agent_payload, art / "daily_event_explanations.json")
save_parquet(event_detail_export, art / "event_detail_export.parquet")
save_json(event_detail_export.to_dict(orient="records"), art / "event_detail_export.json")
save_csv(event_quality_warnings, art / "event_quality_warnings.csv")

mode_and_readiness = pd.DataFrame([
    {"metric": "selected_mode", "value": PRIMARY_PRODUCTION_MODE},
    {"metric": "event_code_mode_enabled", "value": int(MODE_FLAGS.get("event_code_mode", False))},
    {"metric": "semantic_text_mode_enabled", "value": int(MODE_FLAGS.get("semantic_text_mode", False))},
    {"metric": "degraded_mode", "value": int(PRIMARY_PRODUCTION_MODE == "degraded_debug_mode")},
])
save_csv(mode_and_readiness, art / "mode_and_readiness.csv")

publishable_event_export = event_detail_export[[
    "event_date", "mode", "event_id", "cluster_id", "coarse_event_label", "event_confidence_score", "impact_score", "novelty_score",
    "source_breadth", "top_orgs", "top_locs", "top_themes", "representative_urls", "event_warnings", "publishable",
]].copy()
publishable_event_export = publishable_event_export.rename(columns={
    "event_date": "date",
    "coarse_event_label": "label",
    "event_confidence_score": "confidence",
    "event_warnings": "warning_flags",
})
publishable_event_export["warning_flags"] = publishable_event_export["warning_flags"].map(lambda x: json.dumps(x) if isinstance(x, list) else "[]")
save_csv(publishable_event_export, art / "daily_event_packets_flat.csv")

safe_mkdir(art)
jsonl_path = art / "agent_ready_daily_event_payload.jsonl"
with jsonl_path.open("w", encoding="utf-8") as f:
    for obj in daily_agent_payload:
        f.write(json.dumps(obj, ensure_ascii=True, default=str) + "\n")
log_artifact(jsonl_path, "jsonl")

summary_payload = {
    "notebook": "01b_gdelt_event_explainer",
    "selected_mode": PRIMARY_PRODUCTION_MODE,
    "n_input_rows": int(len(rows)),
    "n_event_rows": int(len(event_clusters_rows)),
    "n_events": int(event_scores["event_id"].nunique()),
    "n_days": int(daily_event_explanations["date"].nunique()),
    "n_publishable_days": int(daily_event_explanations["publishable"].sum()) if "publishable" in daily_event_explanations.columns else 0,
    "n_degraded_days": int((~daily_event_explanations["publishable"]).sum()) if "publishable" in daily_event_explanations.columns else int(len(daily_event_explanations)),
    "event_code_mode_enabled": bool(MODE_FLAGS.get("event_code_mode", False)),
    "semantic_text_mode_enabled": bool(MODE_FLAGS.get("semantic_text_mode", False)),
    "artifacts_count": len(ARTIFACT_LOG),
}
save_json(summary_payload, art / "event_notebook_summary.json")

import matplotlib.pyplot as plt

plt.figure(figsize=(10, 4))
daily_event_explanations.set_index("date")["n_events"].plot()
plt.title("Daily Event Count")
plt.ylabel("n_events")
plt.tight_layout()
plt.savefig(plots_dir / "daily_event_count.png", dpi=150)
plt.close()

plt.figure(figsize=(10, 4))
daily_event_explanations.set_index("date")["overall_event_balance"].plot()
plt.title("Daily Overall Event Balance")
plt.ylabel("balance")
plt.tight_layout()
plt.savefig(plots_dir / "daily_event_balance.png", dpi=150)
plt.close()

report_text = "\n".join([
    "# Event Explainer Snapshot",
    f"Mode: {PRIMARY_PRODUCTION_MODE}",
    f"Input rows: {len(rows):,}",
    f"Event rows: {len(event_clusters_rows):,}",
    f"Unique events: {event_scores['event_id'].nunique():,}",
    f"Unique days: {daily_event_explanations['date'].nunique():,}",
    f"Publishable days: {int(daily_event_explanations['publishable'].sum()) if 'publishable' in daily_event_explanations.columns else 0}",
])
save_markdown(report_text, reports_dir / "event_explainer_snapshot.md")

artifact_index = pd.DataFrame(ARTIFACT_LOG).drop_duplicates().sort_values("artifact")
save_csv(artifact_index, art / "artifact_index.csv")

print("Artifacts exported to:", art)
pretty_table(artifact_index, 80)

Artifacts exported to: c:\Users\user\Desktop\client\outputs\gdelt_event_explainer\artifacts


,artifact,type
48,c:\Users\user\Desktop\client\outputs\gdelt_eve...,jsonl
33,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
19,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
17,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
1,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
3,c:\Users\user\Desktop\client\outputs\gdelt_eve...,json
18,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
41,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv
42,c:\Users\user\Desktop\client\outputs\gdelt_eve...,json
47,c:\Users\user\Desktop\client\outputs\gdelt_eve...,csv


## 21) Final Conclusions

This event explainer captures structured same-day event evidence with deterministic candidate generation, overlap-based clustering, auditable labeling, and transparent scoring.

What it captures well:
- Broad event structure from themes, orgs, locs, actor/action/code metadata
- Daily positive/negative/important event decomposition
- Confidence and coverage-aware summaries

What remains noisy:
- Syndicated content and repeated-domain concentration
- Sparse or missing event-code fields on some rows
- Rule-based labeling ambiguity in mixed-topic clusters

Future fusion targets (outside this notebook):
- Reddit signal engine
- StockTwits signal engine
- Google Trends attention engine
- Multisource consistency and temporal conflict resolution


## 22) How This Feeds the Future Sentiment Agent

The future sentiment agent should read exported artifacts from this notebook, **not** rerun notebook logic interactively.

This notebook provides structured event evidence, not final decisions.

Agent fusion inputs should include:
- GDELT signal engine outputs (forecast notebook)
- GDELT event explanation exports (this notebook)
- Reddit signal engine outputs
- StockTwits signal engine outputs
- Google Trends attention outputs

Core contract: this notebook exports an agent-ready daily JSON package with top positive/negative/neutral-important events, supporting metrics, representative URLs, and confidence/warning fields.


## V2 Improvements Added

- stronger clustering: multi-seed candidates, configurable overlap thresholds, merge-reason diagnostics, connected-components merge within day
- stronger labeling: expanded deterministic rules, multi-field evidence (themes/orgs/locs/codes/actors/actions), rule-hit export fields
- richer event summaries: deterministic one-line and rich summaries with confidence rationale and warning context
- stronger warnings: event-level and daily-level warning systems, plus quality-check pass/warn/fail table
- stronger agent-ready exports: canonical JSON/JSONL payloads with representative evidence, warnings, and dominant entities
- improved forecast linkage: alignment score, disagreement heuristic, and contrast-day exports
- improved quality checks: confidence-range checks, label-coverage threshold checks, impossible-count checks, leakage and duplication checks

In [21]:
# Final validation summary (pipeline-native, restart-safe).
rep_url_coverage = float(event_detail_export["representative_urls"].map(lambda x: isinstance(x, list) and len(x) > 0).mean()) if len(event_detail_export) else 0.0
publishable_events = int(event_detail_export["publishable"].sum()) if "publishable" in event_detail_export.columns else 0
publishable_days = int(daily_event_explanations["publishable"].sum()) if "publishable" in daily_event_explanations.columns else 0
degraded_days = int(len(daily_event_explanations) - publishable_days)
singleton_rate = float((event_clusters_summary["n_rows"] == 1).mean()) if len(event_clusters_summary) else np.nan
dominant_domain_rate = float((event_scores["domain_top_share"].fillna(1.0) > 0.80).mean()) if "domain_top_share" in event_scores.columns else np.nan

row_check = event_detail_export[event_detail_export["representative_urls"].map(lambda x: isinstance(x, list) and len(x) > 0)].head(1)
day_check = daily_event_explanations[["date", "publishable", "n_publishable_events", "top_publishable_share", "top_rep_url_coverage", "top_mean_domains", "day_confidence_mean", "rule_ge2_publishable", "rule_quality_mix"]].head(1)

publishability_validation = pd.DataFrame([
    {"metric": "median_n_publishable_events_per_day", "value": float(daily_event_explanations.loc[daily_event_explanations["publishable"], "n_publishable_events"].median()) if publishable_days else np.nan},
    {"metric": "share_publishable_days_rule_ge2_publishable", "value": float(day_rule_audit["rule_ge2_publishable"].mean()) if len(day_rule_audit) else np.nan},
    {"metric": "share_publishable_days_rule_quality_mix", "value": float(day_rule_audit["rule_quality_mix"].mean()) if len(day_rule_audit) else np.nan},
])

final_validation_summary = pd.DataFrame([
    {"metric": "operating_mode", "value": PRIMARY_PRODUCTION_MODE},
    {"metric": "representative_url_coverage", "value": rep_url_coverage},
    {"metric": "publishable_events", "value": publishable_events},
    {"metric": "publishable_days", "value": publishable_days},
    {"metric": "degraded_days", "value": degraded_days},
    {"metric": "singleton_rate", "value": singleton_rate},
    {"metric": "dominant_domain_rate", "value": dominant_domain_rate},
])

save_csv(final_validation_summary, CFG.OUTPUT_DIR / "artifacts" / "final_validation_summary.csv")
save_csv(publishability_validation, CFG.OUTPUT_DIR / "artifacts" / "final_publishability_validation.csv")
save_csv(row_check[["event_date", "event_id", "representative_urls"]], CFG.OUTPUT_DIR / "artifacts" / "final_row_level_rep_url_check.csv")
save_csv(day_check, CFG.OUTPUT_DIR / "artifacts" / "final_day_level_publishability_check.csv")

print("Final validation summary:")
print(final_validation_summary.to_string(index=False))
print("\nPublishability validation:")
print(publishability_validation.to_string(index=False))
print("\nRow-level representative URL check:")
print(row_check[["event_date", "event_id", "representative_urls"]].to_string(index=False))
print("\nDay-level publishability check:")
print(day_check.to_string(index=False))

Final validation summary:
                     metric               value
             operating_mode GKG_structured_mode
representative_url_coverage                 1.0
         publishable_events                3204
           publishable_days                1032
              degraded_days                 777
             singleton_rate            0.651521
       dominant_domain_rate            0.692082

Publishability validation:
                                     metric    value
        median_n_publishable_events_per_day 2.000000
share_publishable_days_rule_ge2_publishable 0.570481
    share_publishable_days_rule_quality_mix 0.000000

Row-level representative URL check:
               event_date           event_id                                                                                                                                                          representative_urls
2021-01-01 00:00:00+00:00 E20210101_1375f817 [http://www.haberlerdetayi.com/2021/01/01/yurticin